# Agente de avaliação de imóvel — dois loops

**Loop 1 (agente)**: o preditor ConvNeXt classifica as fotos (ambiente interna/externa e desgaste em 4 classes),
o `subgrafo_validacao` confere os dados informados (LLM, como em `src/agente_validacao`) e o
`subgrafo_potencial_flip` emite o veredito (LLM).

**Loop 2 (verificação, determinístico)**: três graders checam a saída:
- **G1** imagem: ambiente e desgaste em classes válidas, confianças em [0, 1].
- **G2** coerência: a nota de conservação cai na faixa do desgaste previsto.
- **G3** completude: validação e flip presentes, veredito válido, justificativa e score preenchidos.

Se algum grader falhar, há **uma** nova tentativa do flip com o feedback dos graders.
A imagem não muda entre tentativas (o preditor é determinístico); o retry serve para corrigir o veredito.

Saída: `AvaliacaoImovel`, no formato de `AnalisePotencialFlip` com campos de imagem e de validação.


In [1]:
import asyncio
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

os.environ.setdefault("IMAGENS_JULGADOR", "classificador")
os.environ.setdefault("IMAGENS_CLASSIFICADOR_FOLDS", "5")
os.environ.setdefault("IMAGENS_CLASSIFICADOR_MAX_FOTOS", "0")  # 0 = todas as fotos

# LLM do agente: NVIDIA por padrao (modelo_llm.criar_modelo_flip usa
# nvidia/nemotron-3.5-bolt-e4b e NVIDIA_API_KEY do .env, carregado na chamada).
# Para trocar de provider, descomente e ajuste:
# os.environ["FLIP_BASE_URL"] = "https://openrouter.ai/api/v1"
# os.environ["FLIP_MODEL"] = "openai/gpt-oss-120b"
# os.environ["FLIP_API_KEY"] = os.environ.get("OPENROUTER_API_KEY", "")
# os.environ.setdefault("FLIP_MAX_TOKENS", "4096")
# Groq (sem TPD grande): base https://api.groq.com/openai/v1, modelo openai/gpt-oss-20b.

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados").is_dir()), Path.cwd())
load_dotenv(REPO / ".env", override=False)

from shared.serialization import converter_numpy
from agente_avaliacao_imagens.julgador_classificador import julgar_fotos, LABELS, AMBIENTES
from agente_validacao.subgrafo_validacao import subgrafo_validacao, SubgrafoValidacaoState
from agente_potencial_flip.grafo_potencial_flip import subgrafo_potencial_flip, GrafoPotencialFlipState

df = pd.read_parquet(REPO / "dados" / "joinville" / "joinville_imoveis_limpo_2026-10.parquet")


def _n_fotos(f):
    if isinstance(f, str):
        return 0
    try:
        return len(f)
    except TypeError:
        return 0


candidatos = df[
    (df["fotos"].apply(_n_fotos) >= 3)
    & (pd.to_numeric(df.get("valor_imovel"), errors="coerce") > 300_000)
]


def montar_payload(linha):
    return {
        "fotos_urls": list(linha["fotos"]),
        "dados_imovel": converter_numpy({
            "metragem": linha.get("metragem", 0),
            "banheiros": linha.get("banheiros", 0),
            "vagas": linha.get("vagas", 0),
            "quartos": linha.get("quartos", 0),
            "valor_imovel": linha.get("valor_imovel", 0),
            "bairro": str(linha.get("bairro", "") or ""),
            "tipo_imovel": str(linha.get("tipo_imovel", "") or ""),
            "valor_predito": linha.get("valor_predito", 0),
            "p50_bairro": linha.get("p50_bairro", 0),
            "preco_por_m2": linha.get("preco_por_m2", 0),
        }),
        "descricao_texto": str(linha.get("descricao", "") or ""),
    }


CASO_OK_1 = montar_payload(candidatos.iloc[10])
CASO_OK_2 = montar_payload(candidatos.iloc[20])
CASO_ERRADO = montar_payload(candidatos.iloc[30])
# Erro proposital: valor do imovel multiplicado por 5 para a validacao detectar.
CASO_ERRADO["dados_imovel"]["valor_imovel"] = CASO_ERRADO["dados_imovel"]["valor_imovel"] * 5

# Caso de oportunidade: compra 50% abaixo do ARV. Com reforma de 1500/m2 e
# reserva de 10% do ARV, so passa da regra de 15% comprando ~40% abaixo do ARV;
# sem este caso nenhum run exercita o veredito "True" nem o ramo positivo do G4.
# A descricao acompanha o preco de proposito, para a validacao nao "corrigir"
# o valor de volta (como acontece com o imovel_30_valor_x5).
_d10 = CASO_OK_1["dados_imovel"]
_arv10 = max(_d10.get("valor_predito") or 0,
             (_d10.get("p50_bairro") or 0) * (_d10.get("metragem") or 0))
_preco_barato = round(_arv10 * 0.5, 2)
CASO_BARATO = {
    "fotos_urls": CASO_OK_1["fotos_urls"],
    "dados_imovel": {**_d10, "valor_imovel": _preco_barato},
    "descricao_texto": (
        f"Apartamento de {_d10.get('metragem')} m2, {_d10.get('quartos')} quartos, "
        f"{_d10.get('banheiros')} banheiro e {_d10.get('vagas')} vaga em "
        f"{_d10.get('bairro')}, a venda por R$ {round(_preco_barato)}. "
        "Precisa de atualizacao de piso e pintura."
    ),
}

CASOS = {
    "imovel_10": CASO_OK_1,
    "imovel_20": CASO_OK_2,
    "imovel_30_valor_x5": CASO_ERRADO,
    "imovel_40_barato": CASO_BARATO,
}
print("casos:", list(CASOS))

c:\Users\jefer\Documents\Ciencia-de-dados\Preco-Imoveis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jefer\Documents\Ciencia-de-dados\Preco-Imoveis\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:250: UserWarning: Found nvidia/nemotron-3.5-bolt-e4b in available_models, but type is unknown and inference may fail.
  warnings.warn(


casos: ['imovel_10', 'imovel_20', 'imovel_30_valor_x5', 'imovel_40_barato']


In [2]:
CASOS

{'imovel_10': {'fotos_urls': ['https://resizedimgs.zapimoveis.com.br/img/vr-listing/b41c582e88343fcedef6e12f134e8726/apartamento-com-2-quartos-a-venda-46m-no-pirabeiraba-joinville.webp?action=fit-in&dimension=870x707',
   'https://resizedimgs.zapimoveis.com.br/img/vr-listing/41ca621be127297de3b9441571c860ec/apartamento-com-2-quartos-a-venda-46m-no-pirabeiraba-joinville.webp?action=fit-in&dimension=870x707&seo=false',
   'https://resizedimgs.zapimoveis.com.br/img/vr-listing/9e2b3ef09b4ab4a274d7bccdc7269098/apartamento-com-2-quartos-a-venda-46m-no-pirabeiraba-joinville.webp?action=fit-in&dimension=870x707&seo=false',
   'https://resizedimgs.zapimoveis.com.br/img/vr-listing/9f69e4765f6d925c208949724d440328/apartamento-com-2-quartos-a-venda-46m-no-pirabeiraba-joinville.webp?action=fit-in&dimension=870x707&seo=false',
   'https://resizedimgs.zapimoveis.com.br/img/vr-listing/6317286d41502b28c5692ee1b0c524c9/apartamento-com-2-quartos-a-venda-46m-no-pirabeiraba-joinville.webp?action=fit-in&dim

In [3]:
import os
from typing import Any, Dict, List, Literal, Optional

from pydantic import BaseModel

from agente_potencial_flip.tools_flip import ROI_MINIMO, custo_reforma_m2


class AvaliacaoImovel(BaseModel):
    # Mesmo formato de AnalisePotencialFlip
    score_potencial_flip: float
    potencial_house_flip: Literal["True", "False"]
    justificativa_potencial: str
    riscos: List[str]
    recomendacoes: List[str]
    observacoes: str
    # Imagem (preditor)
    ambiente_fotos: str
    confianca_ambiente: float
    conservacao_fotos: str
    confianca_conservacao: float
    nota_conservacao: float
    # Validacao dos dados
    dados_consistentes: bool
    inconsistencias: List[str]
    possui_erros: Literal["True", "False"]
    # Loop 2
    aprovado_graders: bool
    tentativas: int
    graders: Dict[str, Any]


# Faixa de nota 0-10 esperada para cada desgaste previsto (limite superior exclusivo, exceto 10).
FAIXA_NOTA = {
    "bem conservado": (8.0, 10.0),
    "desgaste leve": (6.0, 8.0),
    "desgastado": (3.0, 6.0),
    "precário": (0.0, 3.0),
}

# Tolerancia de borda do G2. A nota e a media ponderada das 4 classes e a classe
# e o argmax: 5.9 em "desgaste leve" [6.0, 8.0) e o argmax vizinho, nao o agente
# inventando numero. Sem tolerancia o G2 reprova por arredondamento de faixa.
TOLERANCIA_G2 = float(os.environ.get("FLIP_TOLERANCIA_G2", "0.3"))


def _num(valor, default: float = 0.0) -> float:
    """float seguro para os graders: None/str invalida/NaN/inf -> default."""
    try:
        f = float(valor)
    except (TypeError, ValueError):
        return default
    return default if f != f or f in (float("inf"), float("-inf")) else f


def avaliar_graders(imagem, flip, validacao, metricas=None) -> Dict[str, Any]:
    erros = {"G1": [], "G2": [], "G3": [], "G4": [], "G5": [], "G6": []}

    # G1/G2: imagem valida e conservacao coerente
    # Quando o classificador nao tem pesos ou fotos (fallback), imagem traz
    # ambiente='desconhecido' e classe_conservacao=None/'indefinido'.
    # Nesses casos, G1 e G2 nao travam a aprovacao do imovel.
    imagem_fallback = (
        imagem is None
        or getattr(imagem, "ambiente", None) == "desconhecido"
        or getattr(imagem, "classe_conservacao", None) in (None, "indefinido", "")
        or getattr(imagem, "problemas_visiveis", []) == ["Sem fotos para o classificador"]
    )

    if imagem is None:
        erros["G1"].append("analise de imagens ausente")
    elif not imagem_fallback:
        if imagem.ambiente not in AMBIENTES:
            erros["G1"].append(f"ambiente invalido: {imagem.ambiente}")
        if imagem.classe_conservacao not in LABELS:
            erros["G1"].append(f"conservacao invalida: {imagem.classe_conservacao}")
        if not 0 <= imagem.confianca_ambiente <= 1:
            erros["G1"].append(f"confianca_ambiente fora de [0,1]: {imagem.confianca_ambiente}")
        if not 0 <= imagem.confianca_conservacao <= 1:
            erros["G1"].append(f"confianca_conservacao fora de [0,1]: {imagem.confianca_conservacao}")

        # G2: nota coerente com a conservacao prevista
        if imagem.classe_conservacao in FAIXA_NOTA:
            lo, hi = FAIXA_NOTA[imagem.classe_conservacao]
            nota = imagem.score_conservacao
            if not (lo - TOLERANCIA_G2 <= nota and (nota < hi + TOLERANCIA_G2 or hi >= 10)):
                erros["G2"].append(
                    f"nota {nota} fora da faixa de '{imagem.classe_conservacao}' "
                    f"[{lo}, {hi}) +/- {TOLERANCIA_G2}"
                )

    # G3: completude da validacao e do flip, com checagem de coerencia
    if validacao is None:
        erros["G3"].append("validacao ausente")
    if flip is None:
        erros["G3"].append("analise de flip ausente")
    else:
        pot = getattr(flip, "potencial_house_flip", None)
        if pot not in ("True", "False"):
            erros["G3"].append(f"potencial_house_flip invalido: {pot}")
        just = str(getattr(flip, "justificativa_potencial", "") or "").strip()
        if not just or len(just) < 15:
            erros["G3"].append("justificativa vazia ou excessivamente curta")
        import re
        if re.search(r"(?i)^(iniciando|aguardando|chamando|etapa \d|vou |recalculando)", just):
            erros["G3"].append(f"justificativa de etapa em andamento (incompleta): '{just[:60]}...'")

        score = _num(getattr(flip, "score_potencial_flip", None))
        if not 0 <= score <= 10:
            erros["G3"].append(f"score fora de [0,10]: {score}")

        if pot == "True" and score < 5.0:
            erros["G3"].append(f"incoerencia: potencial_house_flip=True mas score={score} (< 5.0)")
        if pot == "False" and score >= 7.0:
            erros["G3"].append(f"incoerencia: potencial_house_flip=False mas score={score} (>= 7.0)")

    # G4: a regra de 15% vem das metricas deterministicas, nao do modelo.
    if metricas is None:
        erros["G4"].append("metricas_flip ausentes: veredito nao pode ser conferido")
    else:
        passa = bool(metricas.get("passa_regra_15pct"))
        roi = metricas.get("roi_conservador")
        if flip is not None and flip.potencial_house_flip == "True" and not passa:
            erros["G4"].append(
                f"aprovou flip com passa_regra_15pct=false (roi_conservador={roi})"
            )

    # G5: a conta da reforma tem que ser a regra vigente e entrar no lucro.
    if metricas is None:
        erros["G5"].append("metricas_flip ausentes: a conta da reforma nao pode ser conferida")
    else:
        metragem = _num(metricas.get("metragem_usada") or metricas.get("metragem"))
        reforma = _num(metricas.get("custo_reforma_estimado"))
        custo = _num(metricas.get("custo_reforma_por_m2")) or custo_reforma_m2()
        if not metragem:
            erros["G5"].append("metragem ausente: reforma nao pode ser conferida")
        else:
            if abs(reforma - custo * metragem) > 0.02:
                erros["G5"].append(
                    f"custo_reforma_estimado {reforma} != {custo}/m2 x {metragem}"
                )
            esperado = round(
                _num(metricas.get("arv_estimado")) * 0.90
                - _num(metricas.get("valor_imovel"))
                - reforma,
                2,
            )
            if abs(esperado - _num(metricas.get("lucro_projetado"))) > 0.02:
                erros["G5"].append(
                    f"lucro_projetado {metricas.get('lucro_projetado')} != {esperado} "
                    "(reforma nao descontada)"
                )
            arv = _num(metricas.get("arv_estimado"))
            if arv:
                teto = round((arv * 0.90 - reforma) / (1 + ROI_MINIMO), 2)
                if abs(teto - _num(metricas.get("preco_maximo_para_regra"))) > 0.02:
                    erros["G5"].append(
                        f"preco_maximo_para_regra "
                        f"{metricas.get('preco_maximo_para_regra')} != {teto}"
                    )

    # G6: as metricas tem que vir da base validada, nao do JSON bruto do modelo.
    if metricas is None:
        erros["G6"].append("metricas ausentes: nao ha como provar a base validada")
    elif validacao is not None:
        base = metricas.get("base_dados")
        if base not in ("validado", "validado_sem_correcao"):
            erros["G6"].append(
                f"metricas nao usaram a base validada (base_dados={base!r})"
            )

    passou = {k: not v for k, v in erros.items()}
    return {"passou": passou, "erros": erros, "ok": all(passou.values())}


def graders_sem_pecas(sessao) -> Dict[str, Any]:
    """Graders de um run incompleto: cada ausencia no seu proprio grader.

    O bloco antigo acusava "sem saida do preditor" sempre que faltasse
    qualquer peca, inclusive com a imagem presente - o Loop 4 aprendia a causa
    errada da falha.
    """
    erros = {"G1": [], "G2": [], "G3": [], "G4": [], "G5": [], "G6": []}
    if sessao.get("imagem") is None:
        erros["G1"].append("classificar_fotos_tool nao rodou (nem o fallback deterministico)")
    if sessao.get("validacao") is None:
        erros["G6"].append("validar_dados_tool nao foi chamada")
    if sessao.get("metricas_tool") is None:
        erros["G4"].append("calcular_metricas_tool nao foi chamada: veredito nao conferido")
        erros["G5"].append("sem metricas: custo de reforma nao conferido")
    if sessao.get("flip") is None:
        erros["G3"].append("sem veredito estruturado")
    motivo = sessao.get("erro_etapa") or sessao.get("provedor_caiu")
    if motivo:
        erros["G3"].append(f"etapa interrompida pelo provedor: {motivo[:160]}")
    passou = {k: not v for k, v in erros.items()}
    return {"passou": passou, "erros": erros, "ok": all(passou.values())}


def montar_saida(imagem, flip, validacao, graders, tentativas, historico) -> AvaliacaoImovel:
    return AvaliacaoImovel(
        score_potencial_flip=round(min(10.0, max(0.0, float(flip.score_potencial_flip))), 2) if flip else 0.0,
        potencial_house_flip=flip.potencial_house_flip if flip else "False",
        justificativa_potencial=flip.justificativa_potencial if flip else "flip indisponivel",
        riscos=list(flip.riscos) if flip else [],
        recomendacoes=list(flip.recomendacoes) if flip else [],
        observacoes=flip.observacoes if flip else "",
        ambiente_fotos=imagem.ambiente,
        confianca_ambiente=imagem.confianca_ambiente,
        conservacao_fotos=imagem.classe_conservacao or "indefinido",
        confianca_conservacao=imagem.confianca_conservacao,
        nota_conservacao=imagem.score_conservacao,
        dados_consistentes=bool(getattr(validacao, "dados_consistentes", False)),
        inconsistencias=list(getattr(validacao, "inconsistencias_encontradas", []) or []),
        possui_erros=getattr(validacao, "possui_erros", "False") or "False",
        aprovado_graders=graders["ok"],
        tentativas=tentativas,
        graders={"passou": graders["passou"], "historico": historico},
    )

In [4]:
async def avaliar_imovel_loop(payload: Dict[str, Any], max_tentativas: int = 1) -> AvaliacaoImovel:
    # Loop 1: imagem (preditor), validacao (LLM) e flip (LLM).
    imagem = await asyncio.to_thread(julgar_fotos, payload["fotos_urls"])

    res_val = await subgrafo_validacao.ainvoke(SubgrafoValidacaoState(
        dados_imovel=payload["dados_imovel"],
        descricao_texto=payload["descricao_texto"],
        api_key=None,
        max_tentativas=0,
    ))
    validacao = res_val.get("analise")

    # Loop 2: graders determinísticos; uma nova tentativa do flip com o feedback.
    feedback = None
    historico = []
    flip = None
    graders = None
    for tentativa in range(1, max_tentativas + 2):
        res_flip = await subgrafo_potencial_flip.ainvoke(GrafoPotencialFlipState(
            dados_imovel=payload["dados_imovel"],
            analise_imagens=imagem,
            analise_validacao=validacao,
            api_key=None,
            feedback=feedback,
            max_tentativas=0,
        ))
        flip = res_flip.get("analise")
        graders = avaliar_graders(imagem, flip, validacao)
        historico.append({"tentativa": tentativa, "ok": graders["ok"], "erros": graders["erros"]})
        if graders["ok"]:
            break
        feedback = "; ".join(e for lista in graders["erros"].values() for e in lista)

    return montar_saida(imagem, flip, validacao, graders, len(historico), historico)

In [5]:
"""resultados = {}
for nome, payload in CASOS.items():
    resultados[nome] = await avaliar_imovel_loop(payload, max_tentativas=1)

tabela = pd.DataFrame([
    {
        "imovel": nome,
        "ambiente": r.ambiente_fotos,
        "conservacao": r.conservacao_fotos,
        "nota": r.nota_conservacao,
        "dados_ok": r.dados_consistentes,
        "possui_erros": r.possui_erros,
        "flip": r.potencial_house_flip,
        "score": r.score_potencial_flip,
        "G1": r.graders["passou"]["G1"],
        "G2": r.graders["passou"]["G2"],
        "G3": r.graders["passou"]["G3"],
        "validou": r.graders.get("validou"),
        "motivo": (r.graders.get("erro_etapa") or r.graders.get("provedor_caiu") or ""),
        "tentativas": r.tentativas,
        "aprovado": r.aprovado_graders,
    }
    for nome, r in resultados.items()
])
print(tabela.to_string(index=False))

print("\nJSON do caso com erro proposital:")
print(resultados["imovel_30_valor_x5"].model_dump_json(indent=2)[:1500])"""

'resultados = {}\nfor nome, payload in CASOS.items():\n    resultados[nome] = await avaliar_imovel_loop(payload, max_tentativas=1)\n\ntabela = pd.DataFrame([\n    {\n        "imovel": nome,\n        "ambiente": r.ambiente_fotos,\n        "conservacao": r.conservacao_fotos,\n        "nota": r.nota_conservacao,\n        "dados_ok": r.dados_consistentes,\n        "possui_erros": r.possui_erros,\n        "flip": r.potencial_house_flip,\n        "score": r.score_potencial_flip,\n        "G1": r.graders["passou"]["G1"],\n        "G2": r.graders["passou"]["G2"],\n        "G3": r.graders["passou"]["G3"],\n        "validou": r.graders.get("validou"),\n        "motivo": (r.graders.get("erro_etapa") or r.graders.get("provedor_caiu") or ""),\n        "tentativas": r.tentativas,\n        "aprovado": r.aprovado_graders,\n    }\n    for nome, r in resultados.items()\n])\nprint(tabela.to_string(index=False))\n\nprint("\nJSON do caso com erro proposital:")\nprint(resultados["imovel_30_valor_x5"].model_

## Loop 1 — agente profundo (`create_deep_agent`)

Mesmo formato do `agente_react_flip` do repo, mas com o **preditor de imagens** no lugar
da descrição por LLM:

```
supervisor "flip-investidor"   tools: think_tool, registrar_decisao
  ├── subagente "fotografo"    classificar_fotos_tool (ConvNeXt) + think_tool
  └── subagente "dados"        metricas_flip_tool, validar_dados_tool + think_tool
```

- `response_format=AnalisePotencialFlip` → o veredito volta estruturado.
- `ExigePensamento(vigiadas=("task",))` → cada delegação a subagente exige `think_tool` antes.
- `interrupt_on={"registrar_decisao": True}` → gravação do veredito exige aprovação humana.
- `MODELO_AGENTE`: `FLIP_MODEL_ID` (ex.: `nvidia:nvidia/nemotron-3.5-bolt-e4b`) ou, por padrão,
  `criar_modelo_flip()` (OpenRouter, que respondeu em ~1 s nas execuções de hoje).

In [6]:
# Ferramentas do agente do flip (sem subagentes, sem task).
import asyncio
import json
import os
import time
from pathlib import Path

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, HumanInTheLoopMiddleware
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver

from agente_avaliacao_imagens.config import CLASSIFICADOR_POR_FOTO, CLASSIFICADOR_SAIDA_COMPLETA
from agente_avaliacao_imagens.julgador_classificador import (
    julgar_fotos,
    julgar_fotos_detalhado,
)
from agente_potencial_flip.modelo_llm import criar_modelo_flip
from agente_potencial_flip.schemas import AnalisePotencialFlip
from agente_potencial_flip.tools_flip import (
    explicar_metricas,
    metricas_flip_tool,
    think_tool,
)
from shared.serialization import converter_numpy
from agente_potencial_flip.trace_flip import ExigePensamento

# A celula pode ser executada antes da c0de5002 (que define REPO): ancora local.
REPO = globals().get("REPO") or next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados").is_dir()),
    Path.cwd(),
)

ARQ_RUNS = REPO / "dados" / "agente_avaliacoes" / "runs.parquet"
ARQ_PROPOSTA = REPO / "dados" / "agente_avaliacoes" / "proposta_harness.md"

ESTADO_AGENTE = {}
SESSOES = {}

MODELO_AGENTE = os.environ.get("FLIP_MODEL_ID", "").strip() or criar_modelo_flip()
print("modelo do agente:", MODELO_AGENTE if isinstance(MODELO_AGENTE, str)
      else type(MODELO_AGENTE).__name__)


# O modelo pequeno (nemotron e4b) entra em laco repetindo a mesma ferramenta
# ate o limite de recursao. Cada resposta carrega o proximo passo obrigatorio, e
# a repeticao ganha um aviso: o empurrao vai junto do dado, em vez de so negar
# o dado (negar deixava o modelo sem saida e ele insistia).
PROXIMO_PASSO = {
    "classificar_fotos_tool": "validar_dados_tool",
    "validar_dados_tool": "calcular_metricas_tool",
    "calcular_metricas_tool": "think_tool",
    "think_tool": "registrar_decisao",
}


def _marcar(nome: str, saida: dict) -> dict:
    """Acrescenta o proximo passo e, se a tool ja rodou, o aviso anti-laco."""
    chave = f"_n_{nome}"
    n = ESTADO_AGENTE.get(chave, 0) + 1
    ESTADO_AGENTE[chave] = n
    proximo = PROXIMO_PASSO.get(nome)
    if proximo:
        saida["proximo_passo"] = proximo
    if n > 1:
        saida["aviso"] = (
            f"{nome} ja rodou {n}x nesta sessao e o resultado e o mesmo. "
            f"NAO repita: chame {proximo} agora."
        )
    return saida


@tool
async def classificar_fotos_tool(
    fotos_urls: list[str] | None = None, refazer: bool = False
) -> str:
    """Classifica as fotos com o preditor ConvNeXt: ambiente (interna/externa) e conservacao (4 classes).

    Com IMAGENS_CLASSIFICADOR_POR_FOTO=true devolve tambem uma entrada por foto.
    Sem argumento, usa as fotos do anuncio em avaliacao. Repetir a chamada para
    o mesmo conjunto devolve cache ("fonte": "cache"): nunca reclassifica 30+
    fotos por engano. `refazer=True` forca nova predicao.
    """
    urls = list(fotos_urls or []) or list(
        (ESTADO_AGENTE.get("payload") or {}).get("fotos_urls") or []
    )
    if not urls:
        return json.dumps({"erro": "sem fotos no anuncio"}, ensure_ascii=False)

    reaproveita = (
        not refazer
        and ESTADO_AGENTE.get("imagem_resumo") is not None
        and tuple(urls) == ESTADO_AGENTE.get("imagem_urls")
    )
    if reaproveita:
        cache = dict(ESTADO_AGENTE["imagem_resumo"])
        cache["fonte"] = "cache"
        return json.dumps(_marcar("classificar_fotos_tool", cache), ensure_ascii=False)

    if not CLASSIFICADOR_POR_FOTO:
        imagem = await asyncio.to_thread(julgar_fotos, urls)
        ESTADO_AGENTE["imagem"] = imagem
        ESTADO_AGENTE["imagem_urls"] = tuple(urls)
        ESTADO_AGENTE["imagem_resumo"] = imagem.resumo()
        if CLASSIFICADOR_SAIDA_COMPLETA:
            return imagem.model_dump_json()
        saida = _marcar("classificar_fotos_tool", dict(imagem.resumo()))
        saida["fonte"] = "preditor"
        return json.dumps(saida, ensure_ascii=False)

    detalhe = await asyncio.to_thread(julgar_fotos_detalhado, urls)
    agregado = detalhe["agregado"]
    ESTADO_AGENTE["imagem"] = agregado
    ESTADO_AGENTE["imagem_por_foto"] = detalhe["por_foto"]
    ESTADO_AGENTE["imagem_urls"] = tuple(urls)
    ESTADO_AGENTE["imagem_resumo"] = agregado.resumo()
    bruto = {
        "por_foto": detalhe["por_foto"],
        "agregado": agregado.model_dump() if CLASSIFICADOR_SAIDA_COMPLETA else agregado.resumo(),
        "fonte": "preditor",
    }
    return json.dumps(_marcar("classificar_fotos_tool", bruto), ensure_ascii=False)


@tool
async def validar_dados_tool(
    dados_imovel_json: str = "", descricao_texto: str = "", refazer: bool = False
) -> str:
    """Confere os numeros do anuncio contra a descricao, usando o subgrafo de validacao.

    Sem argumentos, valida o imovel em avaliacao (payload atual). Repetir a
    chamada devolve cache ("fonte": "cache") em vez de gastar outra chamada de
    LLM; `refazer=True` forca nova validacao.
    """
    payload_atual = ESTADO_AGENTE.get("payload") or {}
    explicito = bool(str(dados_imovel_json).strip() or str(descricao_texto).strip())
    em_cache = ESTADO_AGENTE.get("validacao")
    if em_cache is not None and not refazer and not explicito:
        corpo = _marcar("validar_dados_tool", json.loads(em_cache.model_dump_json()))
        corpo["fonte"] = "cache"
        return json.dumps(corpo, ensure_ascii=False)
    if not str(dados_imovel_json).strip():
        dados_imovel_json = json.dumps(
            payload_atual.get("dados_imovel") or {}, ensure_ascii=False, default=str
        )
    if not descricao_texto:
        descricao_texto = str(payload_atual.get("descricao_texto") or "")
    try:
        dados = json.loads(dados_imovel_json)
    except (TypeError, ValueError) as exc:
        return json.dumps({"erro": f"JSON invalido em dados_imovel_json: {exc}"[:200]}, ensure_ascii=False)
    res = await subgrafo_validacao.ainvoke(SubgrafoValidacaoState(
        dados_imovel=dados,
        descricao_texto=descricao_texto or "",
        api_key=None,
        max_tentativas=0,
    ))
    validacao = res.get("analise")
    ESTADO_AGENTE["validacao"] = validacao
    if validacao is None:
        return json.dumps({"erro": "validacao indisponivel"}, ensure_ascii=False)
    corpo = _marcar("validar_dados_tool", json.loads(validacao.model_dump_json()))
    corpo["fonte"] = "validador"
    return json.dumps(corpo, ensure_ascii=False)


def _dados_para_metricas(payload, validacao):
    """Dados do imovel ajustados pela validacao. So metragem e valor_imovel.

    valor_predito/p50_bairro ficam intocados de proposito: se o LLM pudesse
    "corrigi-los", o ROI deixaria de ser deterministico e o G4 perderia o
    sentido (a regra de 15% passaria a depender da opiniao do modelo).
    """
    dados = dict((payload or {}).get("dados_imovel") or {})
    # O modelo nem sempre preenche os campos *_corrigido; quando ele so devolve
    # `dados_corrigidos`, as chaves de la sao a fonte da correcao.
    devolvidos = getattr(validacao, "dados_corrigidos", None) or {}
    campos = {
        "metragem": ("metragem_corrigida", "metragem"),
        "valor_imovel": ("valor_imovel_corrigido", "valor_imovel"),
    }
    corrigidos = {}
    for campo, (atributo, chave) in campos.items():
        valor = getattr(validacao, atributo, None)
        corrigidos[campo] = valor if valor is not None else devolvidos.get(chave)
    aplicadas = {}
    for campo, valor in corrigidos.items():
        try:
            novo = float(valor)
        except (TypeError, ValueError):
            continue
        if novo <= 0 or novo != novo:
            continue
        antigo = dados.get(campo)
        try:
            mudou = antigo is None or abs(float(antigo) - novo) > 0.01
        except (TypeError, ValueError):
            mudou = True
        if mudou:
            aplicadas[campo] = {"de": antigo, "para": novo}
        dados[campo] = novo
    return dados, aplicadas


@tool
def calcular_metricas_tool(dados_imovel_json: str = "", refazer: bool = False) -> str:
    """Calcula preco_por_m2, ARV, custo de reforma, lucro e ROI, com a memoria de calculo.

    Fonte unica dos numeros. Sem argumento, pega o imovel em avaliacao. Se
    validar_dados_tool ja rodou, usa metragem/valor corrigidos; se nao, calcula
    no bruto e marca `base_dados="bruto_sem_validacao"` (o G6 reprova).
    """
    validacao = ESTADO_AGENTE.get("validacao")
    payload = ESTADO_AGENTE.get("payload") or {}
    bruto = str(dados_imovel_json or "").strip()

    em_cache = ESTADO_AGENTE.get("metricas_tool")
    if em_cache is not None and not refazer and not bruto:
        saida = _marcar("calcular_metricas_tool", dict(em_cache))
        saida["fonte"] = "cache"
        return json.dumps(saida, ensure_ascii=False, default=str)
    if bruto:
        try:
            dados = json.loads(bruto)
        except (TypeError, ValueError) as exc:
            return json.dumps({"erro": f"JSON invalido: {exc}"[:200]}, ensure_ascii=False)
        aplicadas, base = {}, "informado"
    elif validacao is not None:
        dados, aplicadas = _dados_para_metricas(payload, validacao)
        base = "validado" if aplicadas else "validado_sem_correcao"
    else:
        # Recusar fez o modelo desistir da etapa em 6 de 10 imoveis: ele recebia
        # o erro e respondia sem numero. Calcular sempre, e marcar a origem para
        # o grader julgar, da o mesmo sinal sem abrir espaco para desistencia.
        dados = dict(payload.get("dados_imovel") or {})
        aplicadas, base = {}, "bruto_sem_validacao"

    detalhe = converter_numpy(explicar_metricas(dados))
    detalhe["base_dados"] = base
    detalhe["correcoes_aplicadas"] = aplicadas
    detalhe["fonte"] = "metricas"
    detalhe["validacao"] = None if validacao is None else {
        "dados_consistentes": getattr(validacao, "dados_consistentes", None),
        "possui_erros": getattr(validacao, "possui_erros", None),
        "inconsistencias": list(getattr(validacao, "inconsistencias_encontradas", []) or []),
    }
    ESTADO_AGENTE["metricas_tool"] = detalhe
    return json.dumps(_marcar("calcular_metricas_tool", detalhe),
                      ensure_ascii=False, default=str)


@tool
def registrar_decisao(potencial_house_flip: str, score_potencial_flip: float, resumo: str) -> str:
    """Grava o veredito final. So executa depois da aprovacao humana (interrupt_on)."""
    registro = {
        "potencial_house_flip": str(potencial_house_flip),
        "score_potencial_flip": float(score_potencial_flip),
        "resumo": resumo,
    }
    ESTADO_AGENTE["decisao"] = registro
    # Devolve o registro em JSON: o runner reconstrói a decisão a partir do
    # histórico da thread, que é a fonte do que foi de fato aprovado/executado.
    return json.dumps({"registrado": registro}, ensure_ascii=False)


class VerificadorGraders(AgentMiddleware):
    """Loop 2 do artigo: grader deterministico anexado ao agente (after_agent).

    Com response_format, o veredito chega em structured_response, nao por tool.
    """

    async def aafter_agent(self, state, runtime):
        flip = None
        if isinstance(state, dict):
            flip = state.get("structured_response")
        flip = flip or ESTADO_AGENTE.get("flip")
        if flip is not None:
            ESTADO_AGENTE["flip"] = flip
        imagem = ESTADO_AGENTE.get("imagem")
        if imagem is None or flip is None:
            return None
        ESTADO_AGENTE["graders"] = avaliar_graders(
            imagem, flip, ESTADO_AGENTE.get("validacao"), ESTADO_AGENTE.get("metricas_tool")
        )
        return None


SYSTEM_FLIP_NB = """Voce e um investidor senior de Fix & Flip.
A mensagem traz apenas o anuncio (FOTOS_URLS, DADOS_IMOVEL_JSON, DESCRICAO).
Imagem, validacao e metricas vem das suas ferramentas, nesta ordem:
 ETAPA 1: classificar_fotos_tool (SEM argumentos). Preditor ConvNeXt: ambiente
          (interna/externa) e conservacao em 4 classes. Nao existe LLM de visao.
          Repetir a chamada devolve cache ("fonte": "cache"): chame UMA vez.
 ETAPA 2: validar_dados_tool (SEM argumentos). Confere os numeros do anuncio
          contra a descricao e devolve metragem/valor corrigidos.
 ETAPA 3: calcular_metricas_tool (SEM argumentos). Usa os dados VALIDADOS e
          devolve o resultado de cada operacao (formula, entradas, resultado).
 ETAPA 4: think_tool conciliando imagem + validacao + metricas.
 ETAPA 5: registrar_decisao com potencial_house_flip, score_potencial_flip e resumo.
          OBRIGATORIA: e a unica forma de gravar o veredito e pedir aprovacao
          humana. Nao responda o veredito estruturado antes dela.
Depois disso, responda com o veredito estruturado.
Regras:
0) Todas as tools funcionam SEM argumentos: usam o imovel em avaliacao. Nao peca URLs nem JSON ao usuario.
0.1) NUNCA repita uma ferramenta que ja devolveu resultado nesta conversa: use o retorno e siga.
     Cada resposta traz `proximo_passo` (e `aviso` quando voce repetiu): obedeca a ele.
     Nao existe task() nem subagente: nunca escreva `task:...` em proxima_acao.
0.2) A ordem 1 -> 2 -> 3 e obrigatoria: sem `calcular_metricas_tool` executada, `registrar_decisao` e DEVOLVIDA sem gravar nada -- responder antes nao registra veredito.
0.3) UMA ferramenta por turno: espere o resultado antes da proxima. Nunca emita
     validar_dados_tool e calcular_metricas_tool no mesmo turno (elas rodam em
     paralelo e as metricas veriam a validacao ainda vazia).
1) score_potencial_flip e OBRIGATORIAMENTE um numero de 0 a 10 (nota, nao percentual).
   potencial_house_flip e exatamente a string "True" ou "False".
2) Use somente os numeros devolvidos por calcular_metricas_tool; nao recalcule ROI, ARV nem preco_por_m2.
3) Sem fotos no anuncio, pule a ETAPA 1 e avalie com numeros e descricao.
4) potencial_house_flip="True" SOMENTE se passa_regra_15pct for true; se for false, responda "False" mesmo que as fotos parecam boas.
5) O custo de reforma e R$ 1.500/m2 (FLIP_CUSTO_REFORMA_M2) e ja vem descontado em lucro_projetado. Nao invente outro valor nem estime escala de reforma por conta propria.
6) Se aparecer "correcoes_aplicadas", os numeros corrigidos pela validacao ja foram usados: nao os substitua.
7) Para recomendar preco de entrada, use `preco_maximo_para_regra` (teto de compra que ainda fecha os 15%) e `margem_vs_arv_pct` do `contexto`. Nao estime outro alvo de negociacao.
Responda em portugues."""


class MenuEnxuto(AgentMiddleware):
    """Tira do menu so a ferramenta que JA rodou; as pendentes continuam livres.

    O corta-circuito anterior removia TODAS as ferramentas ao atingir N
    ToolMessages. Como think_tool conta no total, o modelo que pensava antes de
    cada etapa chegava ao teto com calcular_metricas_tool ainda pendente e era
    forcado a responder sem conta -- o remedio virava a doenca. Retirar apenas o
    que ja foi feito corta a repeticao (que cairia no cache de qualquer jeito)
    sem fechar o caminho das pedras.
    """

    FEITO = {"classificar_fotos_tool": "imagem",
             "validar_dados_tool": "validacao",
             "calcular_metricas_tool": "metricas_tool"}

    async def awrap_model_call(self, request, handler):
        ferramentas = list(request.tools or [])
        if not ferramentas:
            return await handler(request)
        feitas = {nome for nome, chave in self.FEITO.items()
                  if ESTADO_AGENTE.get(chave) is not None}
        if not feitas:
            return await handler(request)
        restantes = [t for t in ferramentas
                     if getattr(t, "name", None) not in feitas]
        if len(restantes) != len(ferramentas):
            logger.info("menu enxuto: %s ja rodaram, saem do menu", sorted(feitas))
            request = request.override(tools=restantes)
        return await handler(request)


class ExigeAnaliseCompleta(AgentMiddleware):
    """Barra registrar_decisao ate a conta de flip existir.

    Em 6 de 10 imoveis o modelo respondeu `potencial_house_flip=True` sem nunca
    chamar calcular_metricas_tool, e o texto do prompt nao segurou. Aqui a
    chamada devolve uma ToolMessage dizendo exatamente o que falta. Como a tool
    de metricas nao recusa mais, obedecer custa uma chamada -- nao vira laco.
    """

    FALTAS = (("metricas_tool", "calcular_metricas_tool"),
              ("validacao", "validar_dados_tool"))

    async def awrap_tool_call(self, request, handler):
        nome = str((request.tool_call or {}).get("name") or "")
        if nome != "registrar_decisao":
            return await handler(request)

        faltando = [rotulo for chave, rotulo in self.FALTAS
                    if ESTADO_AGENTE.get(chave) is None]
        if faltando:
            return ToolMessage(
                content=(
                    "NAO REGISTRADO: faltam " + " e ".join(faltando) + ". Chame "
                    + " e ".join(faltando) + " sem argumentos e repita "
                    "registrar_decisao. Veredito sem conta nao e avaliado."
                ),
                tool_call_id=(request.tool_call or {}).get("id"),
                name=nome,
            )
        return await handler(request)


def _middlewares():
    base = [
        ExigeAnaliseCompleta(),
        MenuEnxuto(),
        ExigePensamento(vigiadas=("registrar_decisao",)),
        VerificadorGraders(),
        HumanInTheLoopMiddleware(interrupt_on={"registrar_decisao": True}),
    ]
    return base


def criar_agente():
    # create_agent (nao deep): o deep injeta um subagente general-purpose e uma
    # tool `task`, que o modelo usava para re-delegar e estourar a recursao.
    # As 5 ferramentas rodam na ordem do prompt; uma depende da outra.
    return create_agent(
        model=MODELO_AGENTE,
        tools=[classificar_fotos_tool, validar_dados_tool,
               calcular_metricas_tool, think_tool, registrar_decisao],
        middleware=_middlewares(),
        system_prompt=SYSTEM_FLIP_NB,
        response_format=AnalisePotencialFlip,
        checkpointer=MemorySaver(),
    )

modelo do agente: ChatOpenAI


In [7]:
# Executor do agente: sessoes com thread, pausa para aprovacao humana e retry dos graders.
import hashlib
import json as _json
import logging
import os
import time

from agente_avaliacao_imagens.schemas import AnaliseImagens
logger = logging.getLogger("agente_nb")

from langgraph.errors import GraphRecursionError
from langgraph.types import Command


def _pensamentos(messages):
    saida = []
    for m in messages or []:
        for tc in (getattr(m, "tool_calls", None) or []):
            if tc.get("name") == "think_tool":
                args = tc.get("args", {}) or {}
                saida.append({
                    "proxima_acao": str(args.get("proxima_acao", "")),
                    "pensamento": str(args.get("pensamento", "")),
                })
    return saida


def _add_pensamentos(sessao, messages):
    """Com checkpointer o historico volta inteiro a cada ainvoke; sem dedupe duplica."""
    vistos = {(p["proxima_acao"], p["pensamento"]) for p in sessao["pensamentos"]}
    for p in _pensamentos(messages):
        chave = (p["proxima_acao"], p["pensamento"])
        if chave not in vistos:
            vistos.add(chave)
            sessao["pensamentos"].append(p)


def _mensagem(payload, feedback=None):
    """So o anuncio: imagem, validacao e metricas vem das ferramentas do agente."""
    msg = "Avalie este imovel para flip.\n" + (
        "FOTOS_URLS=" + _json.dumps(payload["fotos_urls"], ensure_ascii=False) + "\n"
        + "DADOS_IMOVEL_JSON=" + _json.dumps(payload["dados_imovel"], ensure_ascii=False, default=str) + "\n"
        + "DESCRICAO=" + payload["descricao_texto"][:1200]
    )
    if feedback:
        msg += "\nA tentativa anterior foi reprovada pelos graders: " + feedback + ". Refaca a decisao."
    return msg


def _reconstruir_da_thread(sessao, out):
    """Recupera da thread o veredito estruturado e a decisao aprovada.

    Imagem, validacao e metricas sao escritas pelas ferramentas em
    ESTADO_AGENTE (mesmo processo) e lidas de la.
    """
    flip = out.get("structured_response")
    if flip is not None:
        sessao["flip"] = flip

    for msg in out.get("messages", []) or []:
        if getattr(msg, "name", None) != "registrar_decisao":
            continue
        try:
            corpo = _json.loads(msg.content)
        except (TypeError, ValueError):
            continue
        if isinstance(corpo, dict) and corpo.get("registrado"):
            sessao["decisao"] = corpo["registrado"]

    # Ferramentas escrevem no estado compartilhado do processo.
    if ESTADO_AGENTE.get("imagem") is not None:
        sessao["imagem"] = ESTADO_AGENTE["imagem"]
    if ESTADO_AGENTE.get("validacao") is not None:
        sessao["validacao"] = ESTADO_AGENTE["validacao"]
    if ESTADO_AGENTE.get("metricas_tool") is not None:
        sessao["metricas_tool"] = ESTADO_AGENTE["metricas_tool"]
    if sessao.get("decisao") is None and ESTADO_AGENTE.get("decisao") is not None:
        sessao["decisao"] = ESTADO_AGENTE["decisao"]

    # Se o modelo nao chamou a tool de fotos, o preditor e deterministico: roda
    # aqui. Guarda as chaves de cache para a tool nao reclassificar as imagens.
    if sessao.get("imagem") is None:
        urls = (sessao.get("payload") or {}).get("fotos_urls") or []
        if urls:
            logger.warning("agente nao avaliou as fotos; usando o preditor direto")
            imagem = julgar_fotos(list(urls))
            sessao["imagem"] = imagem
            sessao["fallback_imagem"] = True
            ESTADO_AGENTE["imagem"] = imagem
            ESTADO_AGENTE["imagem_urls"] = tuple(urls)
            ESTADO_AGENTE["imagem_resumo"] = imagem.resumo()


def resumo_tools(messages):
    """Pareia tool_calls com as ToolMessages e devolve [{tool, args, saida}].

    Cada chamada do modelo vira uma linha com a saida da ferramenta.
    """
    pendente, linhas = {}, []
    for m in messages or []:
        for tc in (getattr(m, "tool_calls", None) or []):
            pendente[tc.get("id")] = (tc.get("name"), tc.get("args"))
        if type(m).__name__ == "ToolMessage":
            nome, args = pendente.pop(
                getattr(m, "tool_call_id", None), (getattr(m, "name", None), None)
            )
            item = {"tool": nome, "args": args, "saida": str(m.content)[:1500]}
            if linhas and linhas[-1]["tool"] == item["tool"] and linhas[-1]["args"] == item["args"]:
                continue  # chamada repetida em sequencia: nao polui a trilha
            linhas.append(item)
    return linhas


MAX_CHAMADAS_TOOL = int(os.environ.get("FLIP_MAX_CHAMADAS_TOOL", "4"))


def _chamadas_por_tool(messages):
    """Conta chamadas por nome de ferramenta (protecao contra loop)."""
    contagem = {}
    for m in messages or []:
        for tc in (getattr(m, "tool_calls", None) or []):
            nome = str(tc.get("name") or "?")
            contagem[nome] = contagem.get(nome, 0) + 1
    return contagem


PROVEDOR_CAIU = ("429", "502", "503", "rate limit", "throttle", "timeout",
                 "maximum context", "connection", "unavailable")


async def _historico_do_checkpointer(sessao):
    """O historico real vive no checkpointer: e dele que se recupera a trilha.

    Sem isso, uma falha no meio do run devolvia o output anterior (vazio) e o
    log saia com `chamadas={}` mesmo depois de 45 passos de agente.
    """
    try:
        estado = await sessao["agente"].aget_state(sessao["cfg"])
        valores = dict(getattr(estado, "values", None) or {}) if estado else {}
        return valores.get("messages", []) or [], valores.get("structured_response")
    except Exception:  # noqa: BLE001 - recuperar nunca derruba a celula
        anterior = sessao.get("out") or {}
        return anterior.get("messages", []) or [], anterior.get("structured_response")


async def _invocar(sessao, entrada):
    """`ainvoke` com rede de seguranca: nenhuma falha derruba a celula.

    Recursao (laco de ferramentas) e erro do provedor encerram a etapa e viram
    motivo nos graders - o pior caso e um veredito incompleto, nao um traceback.
    """
    try:
        return await sessao["agente"].ainvoke(entrada, config=sessao["cfg"])
    except GraphRecursionError:
        logger.warning("limite de recursao atingido; encerrando a etapa")
        sessao["recursao_atingida"] = True
    except Exception as exc:  # noqa: BLE001 - HTTP 400/429/timeout nao podem matar a celula
        logger.warning("chamada do agente falhou: %s: %s", type(exc).__name__, str(exc)[:300])
        sessao["erro_etapa"] = f"{type(exc).__name__}: {str(exc)[:300]}"
    msgs, flip = await _historico_do_checkpointer(sessao)
    return {"messages": msgs, "structured_response": flip}


async def _resolver_interrupts(sessao, out):
    """Retoma ate acabar as pausas, ou para quando quem decide e o humano."""
    while out.get("__interrupt__"):
        req = out["__interrupt__"][0]
        sessao["solicitacoes"].append(getattr(req, "value", req))
        if sessao["decisao_humana"] is None:
            return out, True
        out = await _invocar(
            sessao, Command(resume={"decisions": [{"type": sessao["decisao_humana"]}]})
        )
    return out, False


def _saida(sessao, graders):
    saida = montar_saida(
        sessao.get("imagem") or julgar_fotos([]),
        sessao.get("flip"),
        sessao.get("validacao"),
        graders,
        len(sessao["historico"]),
        sessao["historico"],
    )
    if sessao.get("imagem") is not None:
        saida.graders["imagem_resumo"] = sessao["imagem"].resumo()
    saida.graders["tools"] = sessao.get("tools", [])
    saida.graders["pensamentos"] = sessao["pensamentos"]
    saida.graders["decisao"] = sessao.get("decisao")
    saida.graders["solicitacoes_aprovacao"] = sessao["solicitacoes"]
    saida.graders["decisao_humana"] = sessao["decisao_humana"]
    saida.graders["resumo_agente"] = sessao.get("ultima_msg", "")
    saida.graders["chamadas_por_tool"] = sessao.get("chamadas")
    saida.graders["limite_chamadas"] = sessao.get("limite_chamadas")
    saida.graders["erro_etapa"] = sessao.get("erro_etapa")
    saida.graders["provedor_caiu"] = sessao.get("provedor_caiu")
    saida.graders["validou"] = sessao.get("validacao") is not None
    saida.graders["classificou"] = sessao.get("imagem") is not None
    saida.graders["etapas_runner"] = sessao.get("recuperadas") or []
    saida.graders["quem_analisou"] = "runner" if sessao.get("recuperadas") else "agente"
    saida.graders["metricas_tool"] = sessao.get("metricas_tool")
    saida.graders["metricas_base"] = (sessao.get("metricas_tool") or {}).get("base_dados")
    return saida


def _span_ferramenta(nome, saida, t0, origem):
    """Marca no LangSmith a etapa que o runner executou FORA do grafo.

    O que roda dentro do grafo (turno do modelo, ToolNode) ja e traceado pelo
    LANGCHAIN_TRACING_V2. `garantir_etapas` roda no runner, entao sem isto a
    recuperacao ficaria invisivel na arvore.
    """
    try:
        from langsmith.run_helpers import trace as ls_trace

        corpo = str(saida)
        imovel = str((ESTADO_AGENTE.get("payload") or {}).get("_id", "?"))
        with ls_trace(f"runner:{nome}", run_type="tool",
                      inputs={"origem": origem, "imovel": imovel},
                      project_name=os.environ.get("LANGCHAIN_PROJECT")) as run:
            run.end(outputs={
                "duracao_ms": round((time.perf_counter() - t0) * 1000, 1),
                "cache": '"fonte": "cache"' in corpo,
                "recusada": corpo.lstrip().startswith('{"erro"'),
                "saida": corpo[:4000],
            })
    except Exception as exc:  # noqa: BLE001
        logger.warning("span de %s no LangSmith falhou: %s", nome, exc)


async def garantir_etapas(sessao):
    """Roda de fora o que o agente pulou.

    Medido no lote: o modelo ou percorre as 5 etapas (~245 s) ou responde em
    ~25 s depois de UMA chamada, sem validar e sem calcular. Pedir de novo nao
    conserta -- ele repete o atalho com o proprio veredito na thread. As etapas
    sao deterministicas de conta (a visao e o ConvNeXt, nao o LLM), entao o
    runner as executa e o veredito e pedido de novo COM os numeros na mensagem.
    """
    recuperadas = []
    quem_antes = ESTADO_AGENTE.get("_quem")
    ESTADO_AGENTE["_quem"] = "runner"
    try:
        # Sem fotos na lista, a ferramenta nunca preenche `imagem` e a
        # recuperacao a chamaria de novo a cada passada.
        tem_fotos = bool((sessao.get("payload") or {}).get("fotos_urls"))
        if ESTADO_AGENTE.get("imagem") is None and tem_fotos:
            t0 = time.perf_counter()
            saida = await classificar_fotos_tool.ainvoke({})
            _span_ferramenta("classificar_fotos_tool", saida, t0, "runner")
            recuperadas.append("classificar_fotos_tool")

        if ESTADO_AGENTE.get("validacao") is None:
            t0 = time.perf_counter()
            saida = await validar_dados_tool.ainvoke({})
            _span_ferramenta("validar_dados_tool", saida, t0, "runner")
            recuperadas.append("validar_dados_tool")

        if ESTADO_AGENTE.get("metricas_tool") is None:
            t0 = time.perf_counter()
            saida = calcular_metricas_tool.invoke({})
            _span_ferramenta("calcular_metricas_tool", saida, t0, "runner")
            recuperadas.append("calcular_metricas_tool")
    finally:
        if quem_antes is None:
            ESTADO_AGENTE.pop("_quem", None)
        else:
            ESTADO_AGENTE["_quem"] = quem_antes

    if recuperadas:
        sessao["imagem"] = sessao.get("imagem") or ESTADO_AGENTE.get("imagem")
        sessao["validacao"] = sessao.get("validacao") or ESTADO_AGENTE.get("validacao")
        sessao["metricas_tool"] = ESTADO_AGENTE.get("metricas_tool")
        sessao["recuperadas"] = recuperadas
        logger.warning("etapas puladas pelo agente, rodadas pelo runner: %s", recuperadas)
    return recuperadas


def _mensagem_com_conta(sessao):
    """Repedido do veredito com os numeros NA mensagem.

    Insistir com texto nao mudava nada: o modelo tinha acabado de responder sem
    conta e a thread dele nao tinha numero algum para reler.
    """
    payload = sessao["payload"]
    conta = {k: v for k, v in (sessao.get("metricas_tool") or {}).items()
             if k not in ("operacoes", "validacao")}
    msg = "Reavalie ESTE imovel e chame registrar_decisao com o veredito.\n"
    msg += "METRICAS_JSON=" + _json.dumps(conta, ensure_ascii=False, default=str) + "\n"
    if sessao.get("imagem") is not None:
        msg += "IMAGEM_JSON=" + _json.dumps(sessao["imagem"].resumo(), ensure_ascii=False) + "\n"
    val = sessao.get("validacao")
    if val is not None:
        msg += "VALIDACAO_JSON=" + _json.dumps({
            "dados_consistentes": getattr(val, "dados_consistentes", None),
            "possui_erros": getattr(val, "possui_erros", None),
            "inconsistencias": list(getattr(val, "inconsistencias_encontradas", []) or []),
        }, ensure_ascii=False, default=str) + "\n"
    msg += ("potencial_house_flip depende de passa_regra_15pct desses numeros; use "
            "preco_maximo_para_regra como alvo de negociacao. Nao chame "
            "classificar_fotos_tool, validar_dados_tool nem calcular_metricas_tool.\n")
    msg += ("FOTOS_URLS=" + _json.dumps(payload["fotos_urls"], ensure_ascii=False) + "\n"
            + "DADOS_IMOVEL_JSON="
            + _json.dumps(payload["dados_imovel"], ensure_ascii=False, default=str) + "\n"
            + "DESCRICAO=" + payload["descricao_texto"][:1200])
    return msg


async def _processar(sessao, out, max_tentativas=1):
    for tentativa in range(1, max_tentativas + 2):
        out, pausado = await _resolver_interrupts(sessao, out)
        if pausado:
            sessao["out"] = out
            return {"estado": "aguardando_humano", "sessao": sessao}

        msgs = out.get("messages", [])
        _add_pensamentos(sessao, msgs)
        sessao["tools"] = resumo_tools(msgs)
        sessao["ultima_msg"] = str(msgs[-1].content) if msgs else ""

        _reconstruir_da_thread(sessao, out)

        chamadas = _chamadas_por_tool(msgs)
        sessao["chamadas"] = chamadas
        estourou = bool(sessao.get("recursao_atingida"))
        repetiu = {n: c for n, c in chamadas.items() if c >= MAX_CHAMADAS_TOOL}
        if repetiu:
            # Informativo: a repeticao cai no cache da ferramenta, entao nao
            # justifica cancelar o retry dos graders.
            sessao["limite_chamadas"] = repetiu
            logger.info("ferramentas repetidas (cache): %s", repetiu)

        # Veredito sem conta nao e veredito: roda o que faltou e pede de novo,
        # agora com os numeros na mensagem. Uma vez por sessao.
        sem_conta = (ESTADO_AGENTE.get("metricas_tool") is None
                     or ESTADO_AGENTE.get("validacao") is None)
        # Recursao atingida NAO e motivo para pular a recuperacao: e justamente o
        # caso do modelo que girou em classificar_fotos_tool ate o teto sem nunca
        # calcular (medido em 3 de 10 do lote). Cada ainvoke tem orçamento novo de
        # passos, entao a conta ainda pode ser feita e o veredito pedido.
        # So nao se tenta quando o provedor respondeu erro: ai seria queimar chamada.
        if (sem_conta and not sessao.get("recuperou_etapas")
                and not sessao.get("erro_etapa")):
            sessao["recuperou_etapas"] = True
            await garantir_etapas(sessao)
            sessao["flip"] = None                # veredito sem base nao vale
            ESTADO_AGENTE.pop("flip", None)      # senao a thread recupera o antigo
            sessao["decisao"] = None
            out = await _invocar(
                sessao,
                {"messages": [{"role": "user", "content": _mensagem_com_conta(sessao)}]},
            )
            out, pausado = await _resolver_interrupts(sessao, out)
            if pausado:
                sessao["out"] = out
                return {"estado": "aguardando_humano", "sessao": sessao}
            _reconstruir_da_thread(sessao, out)
            msgs = out.get("messages", []) or msgs
            sessao["tools"] = resumo_tools(msgs)
            if msgs:
                sessao["ultima_msg"] = str(msgs[-1].content)
            if sessao.get("flip") is None:
                graders = graders_sem_pecas(sessao)
                sessao["historico"].append({"tentativa": tentativa, "ok": False,
                                            "erros": graders["erros"]})
                return {"estado": "concluido", "sessao": sessao,
                        "saida": _saida(sessao, graders)}

        # O modelo costuma emitir o veredito estruturado sem chamar
        # registrar_decisao. Pedir de novo custava um turno e o modelo reiniciava
        # a sequencia de ferramentas (laco). A decisao sai do proprio veredito.
        if sessao.get("decisao") is None and sessao.get("flip") is not None:
            flip = sessao["flip"]
            decisao = {
                "potencial_house_flip": str(flip.potencial_house_flip),
                "score_potencial_flip": float(flip.score_potencial_flip),
                "resumo": str(flip.justificativa_potencial)[:300],
                "origem": "veredito_estruturado",
            }
            sessao["decisao"] = decisao
            ESTADO_AGENTE["decisao"] = decisao
            logger.info("decisao sintetizada do veredito estruturado")

        if (sessao.get("imagem") is None or sessao.get("flip") is None
                or sessao.get("metricas_tool") is None or sessao.get("validacao") is None):
            graders = graders_sem_pecas(sessao)
        else:
            graders = avaliar_graders(
                sessao["imagem"], sessao["flip"], sessao.get("validacao"),
                ESTADO_AGENTE.get("metricas_tool"),
            )
        sessao["historico"].append({"tentativa": tentativa, "ok": graders["ok"], "erros": graders["erros"]})
        # So completa do global o que a thread nao forneceu (nao sobrescreve).
        sessao["imagem"] = sessao.get("imagem") or ESTADO_AGENTE.get("imagem")
        sessao["flip"] = sessao.get("flip") or ESTADO_AGENTE.get("flip")
        sessao["validacao"] = sessao.get("validacao") or ESTADO_AGENTE.get("validacao")
        caiu = any(m in (sessao.get("erro_etapa") or "").lower() for m in PROVEDOR_CAIU)
        if caiu:
            sessao["provedor_caiu"] = sessao["erro_etapa"]
        # Retry nao conserta provedor indisponivel: so gasta chamada e polui o
        # historico com um veredito tardio.
        if estourou and not graders["ok"]:
            logger.warning("estouro de recursao sem veredito valido: %s", chamadas)
        if graders["ok"] or tentativa > max_tentativas or estourou or caiu:
            return {"estado": "concluido", "sessao": sessao, "saida": _saida(sessao, graders)}

        faltam = []
        if sessao.get("validacao") is None:
            faltam.append("validar_dados_tool")
        if ESTADO_AGENTE.get("metricas_tool") is None:
            faltam.append("calcular_metricas_tool")
        if faltam:
            # O texto do grader mostrava o erro e o modelo respondia igual. Aqui a
            # instruicao nomeia a ferramenta que falta e veta as ja feitas.
            feedback = ("Voce respondeu sem rodar " + " e ".join(faltam) + ". Chame "
                        + " e ".join(faltam) + " SEM argumentos agora, e so depois "
                        "chame registrar_decisao. Nao repita classificar_fotos_tool "
                        "nem think_tool.")
        else:
            feedback = (
                "OS GRADERS AUTOMATICOS reprovaram o veredito anterior; nao foi rejeicao humana. "
                + "; ".join(e for lista in graders["erros"].values() for e in lista)
            )
        out = await _invocar(
            sessao,
            {"messages": [{"role": "user", "content": _mensagem(sessao["payload"], feedback)}]},
        )
    return {"estado": "concluido", "sessao": sessao, "saida": _saida(sessao, graders)}


def _cfg_trace(payload, thread_id):
    """Config do ainvoke com o que identifica a trace no LangSmith.

    Sem isto a raiz aparece como `LangGraph`, com `tags=[]` e so um `thread_id`
    com hash: dois imoveis do lote podem cair no mesmo id e nao ha como saber
    qual arvore e de qual anuncio. `tags` e `metadata` viram filtro na UI.
    """
    imovel = str((payload or {}).get("_id") or "solto")
    dados = (payload or {}).get("dados_imovel") or {}
    return {
        "configurable": {"thread_id": thread_id},
        "recursion_limit": int(os.environ.get("FLIP_RECURSION_LIMIT", "45")),
        "run_name": imovel,
        "tags": [f"imovel:{imovel}",
                 f"lote:{os.environ.get('FLIP_LOTE_TAG', 'atiradores')}",
                 "flip"],
        "metadata": {
            "imovel": imovel,
            "thread_id": thread_id,
            "bairro": str(dados.get("bairro") or ""),
            "tipo_imovel": str(dados.get("tipo_imovel") or ""),
            "preco_por_m2": dados.get("preco_por_m2"),
            "metragem": dados.get("metragem"),
            "valor_imovel": dados.get("valor_imovel"),
        },
    }


async def iniciar_avaliacao(payload, thread_id, decisao_humana=None, max_tentativas=1):
    """decisao_humana=None pausa para aprovacao; "approve"/"edit"/"reject" decide sozinho."""
    ESTADO_AGENTE.clear()
    ESTADO_AGENTE["payload"] = payload
    sessao = {
        "thread_id": thread_id,
        "payload": payload,
        "agente": criar_agente(),
        # Teto de turnos: o modelo repete ferramentas e cada turno entra no
        # historico. 60 permitia ~30 chamadas e estourava 131k tokens de contexto.
        "cfg": _cfg_trace(payload, thread_id),
        "decisao_humana": decisao_humana,
        "historico": [], "pensamentos": [], "solicitacoes": [], "tools": [],
    }
    SESSOES[thread_id] = sessao

    # Imagem, validacao e metricas NAO sao pre-calculadas: as ferramentas do
    # agente fazem isso na ordem certa (e as metricas usam os dados validados).
    out = await _invocar(
        sessao,
        {"messages": [{"role": "user", "content": _mensagem(payload)}]},
    )
    sessao["out"] = out
    return await _processar(sessao, out, max_tentativas)


async def retomar_avaliacao(thread_id, decisao="approve", max_tentativas=1):
    sessao = SESSOES[thread_id]
    sessao["decisao_humana"] = decisao
    out = await _invocar(sessao, Command(resume={"decisions": [{"type": decisao}]}))
    sessao["out"] = out
    return await _processar(sessao, out, max_tentativas)


async def avaliar_imovel_agente(payload, max_tentativas=1) -> "AvaliacaoImovel":
    """Atalho sem pausa humana (usado nos lotes e no teste de 3 imoveis)."""
    # O hash sozinho colide entre imoveis parecidos; o _id do lote na frente
    # garante thread e trace distintas por anuncio.
    _marca = str(payload.get("_id") or "")
    thread = (f"auto-{_marca}-" if _marca else "auto-") + hashlib.sha1(
        _json.dumps(payload["dados_imovel"], sort_keys=True, default=str).encode("utf-8")
    ).hexdigest()[:10]
    r = await iniciar_avaliacao(payload, thread, decisao_humana="approve",
                                max_tentativas=max_tentativas)
    if "saida" not in r:
        s = r["sessao"]
        g = {"passou": {"G1": False, "G2": False, "G3": False,
                         "G4": False, "G5": False, "G6": False}, "ok": False,
             "erros": {"G1": [], "G2": [], "G3": ["nao concluido"],
                       "G4": [], "G5": [], "G6": []}}
        return montar_saida(s.get("imagem") or julgar_fotos([]), s.get("flip"), s.get("validacao"),
                            g, len(s["historico"]), s["historico"])
    return r["saida"]

In [8]:
"""resultados_agente = {}
for nome, payload in CASOS.items():
    resultados_agente[nome] = await avaliar_imovel_agente(payload, max_tentativas=1)

tabela_agente = pd.DataFrame([
    {
        "imovel": nome,
        "ambiente": r.ambiente_fotos,
        "conservacao": r.conservacao_fotos,
        "nota": r.nota_conservacao,
        "dados_ok": r.dados_consistentes,
        "possui_erros": r.possui_erros,
        "flip": r.potencial_house_flip,
        "score": r.score_potencial_flip,
        "G1": r.graders["passou"]["G1"],
        "G2": r.graders["passou"]["G2"],
        "G3": r.graders["passou"]["G3"],
        "G4": r.graders["passou"].get("G4"),
        "G5": r.graders["passou"].get("G5"),
        "G6": r.graders["passou"].get("G6"),
        "base_metricas": r.graders.get("metricas_base"),
        "validou": r.graders.get("validou"),
        "motivo": (r.graders.get("erro_etapa") or r.graders.get("provedor_caiu") or ""),
        "pensamentos": len(r.graders["pensamentos"]),
        "tentativas": r.tentativas,
        "aprovado": r.aprovado_graders,
    }
    for nome, r in resultados_agente.items()
])
#print(tabela_agente.to_string(index=False))

primeiro = resultados_agente[list(resultados_agente)[0]]
print("\ntrilha de pensamentos do primeiro imovel:")
for p in primeiro.graders["pensamentos"]:
    print(f"- [{p['proxima_acao']}] {p['pensamento'][:200]}")

print("\nhistorico de graders (erros por tentativa):")
for nome, r in resultados_agente.items():
    for h in r.graders["historico"]:
        if not h["ok"]:
            print(nome, "tentativa", h["tentativa"], "->", {k: v for k, v in h["erros"].items() if v})"""

'resultados_agente = {}\nfor nome, payload in CASOS.items():\n    resultados_agente[nome] = await avaliar_imovel_agente(payload, max_tentativas=1)\n\ntabela_agente = pd.DataFrame([\n    {\n        "imovel": nome,\n        "ambiente": r.ambiente_fotos,\n        "conservacao": r.conservacao_fotos,\n        "nota": r.nota_conservacao,\n        "dados_ok": r.dados_consistentes,\n        "possui_erros": r.possui_erros,\n        "flip": r.potencial_house_flip,\n        "score": r.score_potencial_flip,\n        "G1": r.graders["passou"]["G1"],\n        "G2": r.graders["passou"]["G2"],\n        "G3": r.graders["passou"]["G3"],\n        "G4": r.graders["passou"].get("G4"),\n        "G5": r.graders["passou"].get("G5"),\n        "G6": r.graders["passou"].get("G6"),\n        "base_metricas": r.graders.get("metricas_base"),\n        "validou": r.graders.get("validou"),\n        "motivo": (r.graders.get("erro_etapa") or r.graders.get("provedor_caiu") or ""),\n        "pensamentos": len(r.graders["pe

## Resultado de cada ferramenta do agente

Roda o agente em um imóvel e imprime, em ordem, **cada chamada de ferramenta com o retorno**:
`think_tool`, `task` (com o relatório do subagente), `registrar_decisao`.

As ferramentas internas dos subagentes (`classificar_fotos_tool`, `validar_dados_tool`,
`metricas_flip_tool`) não aparecem nessa lista porque rodam em outro agente; o que o supervisor
enxerga é o relatório do `task`. Os artefatos dessas tools internas ficam em `ESTADO_AGENTE`
e são impressos no fim.

In [9]:
# Resultado de cada ferramenta chamada pelo supervisor.
import json
"""import os

# reduz a reserva por request (o OpenRouter devolve 402 quando o saldo
# nao cobre o pior caso; ler em runtime, sem reiniciar o kernel)
os.environ.setdefault("FLIP_MAX_TOKENS", "1024")

s_tools = await avaliar_imovel_agente(CASOS["imovel_10"], max_tentativas=1)

tools = s_tools.graders.get("tools", [])
print(f"chamadas de ferramenta: {len(tools)}\n")
for i, t in enumerate(tools, 1):
    args = json.dumps(t.get("args") or {}, ensure_ascii=False)
    print(f"{i}. {t.get('tool')}  args={args[:400]}")
    print("   ->", (t.get("saida") or "").replace("\n", " ")[:500])
    print()

print("=== graders ===")
print("limite de chamadas:", s_tools.graders.get("limite_chamadas"))
print("passou:", s_tools.graders.get("passou"), "| tentativas:", s_tools.tentativas,
      "| aprovado:", s_tools.aprovado_graders)
print("historico:", json.dumps(s_tools.graders.get("historico"), ensure_ascii=False, default=str)[:400])

print("\n=== mensagem final do agente ===")
print((s_tools.graders.get("resumo_agente") or "(vazia)")[:800])

print("\n=== artefatos das ferramentas ===")
print("imagem (resumo):", json.dumps(s_tools.graders.get("imagem_resumo"), ensure_ascii=False))
print("metricas (base):", s_tools.graders.get("metricas_base"))
metricas_tool = s_tools.graders.get("metricas_tool") or {}
if metricas_tool:
    print("metricas (tool):", json.dumps(
        {k: v for k, v in metricas_tool.items() if k != "operacoes"}, ensure_ascii=False)[:700])
    print("correcoes aplicadas:", json.dumps(
        metricas_tool.get("correcoes_aplicadas"), ensure_ascii=False))
    print("contexto:", json.dumps(metricas_tool.get("contexto"), ensure_ascii=False))
    print("teto de compra para os 15%:", metricas_tool.get("preco_maximo_para_regra"))
    for op in metricas_tool.get("operacoes", []):
        print(f"  {op['passo']}: {op['formula']} = {op['resultado']}")
else:
    print("metricas (tool): NAO CHAMADA")
print("decisao:", json.dumps(s_tools.graders.get("decisao"), ensure_ascii=False))
print("validacao:", {
    "dados_consistentes": s_tools.dados_consistentes,
    "possui_erros": s_tools.possui_erros,
    "inconsistencias": s_tools.inconsistencias,
})"""

'import os\n\n# reduz a reserva por request (o OpenRouter devolve 402 quando o saldo\n# nao cobre o pior caso; ler em runtime, sem reiniciar o kernel)\nos.environ.setdefault("FLIP_MAX_TOKENS", "1024")\n\ns_tools = await avaliar_imovel_agente(CASOS["imovel_10"], max_tentativas=1)\n\ntools = s_tools.graders.get("tools", [])\nprint(f"chamadas de ferramenta: {len(tools)}\n")\nfor i, t in enumerate(tools, 1):\n    args = json.dumps(t.get("args") or {}, ensure_ascii=False)\n    print(f"{i}. {t.get(\'tool\')}  args={args[:400]}")\n    print("   ->", (t.get("saida") or "").replace("\n", " ")[:500])\n    print()\n\nprint("=== graders ===")\nprint("limite de chamadas:", s_tools.graders.get("limite_chamadas"))\nprint("passou:", s_tools.graders.get("passou"), "| tentativas:", s_tools.tentativas,\n      "| aprovado:", s_tools.aprovado_graders)\nprint("historico:", json.dumps(s_tools.graders.get("historico"), ensure_ascii=False, default=str)[:400])\n\nprint("\n=== mensagem final do agente ===")\npri

## Resultado de cada ferramenta — chamada direta (sem LLM)

Chama as tools do agente **uma por uma**, sem passar pelo modelo, para você ver o retorno cru
de cada uma:

- `classificar_fotos_tool` — preditor ConvNeXt (ambiente + desgaste), sem LLM.
- `metricas_flip_tool` — ROI/ARV determinísticos, sem LLM.
- `validar_dados_tool` — usa LLM no subgrafo de validação; desligado por padrão para não
  travar quando não há cota (`TENTAR_VALIDACAO = True` para ligar).

Útil quando o provedor está sem crédito: as duas primeiras sempre funcionam.

In [10]:
# Resultado de cada ferramenta, sem depender do agente/LLM.
import json
import os

os.environ.setdefault("IMAGENS_CLASSIFICADOR_MAX_FOTOS", "0")  # todas as fotos

payload_tools = CASOS["imovel_10"]
ESTADO_AGENTE.clear()
ESTADO_AGENTE["payload"] = payload_tools

print("fotos do imovel:", len(payload_tools["fotos_urls"]))

print("\n=== 1) classificar_fotos_tool (sem LLM) ===")
saida_fotos = await classificar_fotos_tool.ainvoke({"fotos_urls": []})
print(saida_fotos[:900])

print("\n=== 1b) classificar_fotos_tool repetida (deve vir do cache) ===")
print((await classificar_fotos_tool.ainvoke({"fotos_urls": []}))[:300])

print("\n=== 2) validar_dados_tool (usa LLM; corrige metragem/valor) ===")
TENTAR_VALIDACAO = True  # False: a etapa 3 recusa, de proposito
if TENTAR_VALIDACAO:
    print((await validar_dados_tool.ainvoke({}))[:1200])
else:
    print("desligado (TENTAR_VALIDACAO=False)")

print("\n=== 3) calcular_metricas_tool (sem LLM, usa os dados validados) ===")
bruto_metricas = calcular_metricas_tool.invoke({})
print(bruto_metricas[:1500])

metricas_calc = json.loads(bruto_metricas)
if "operacoes" in metricas_calc:
    print("\n--- memoria de calculo ---")
    for op in metricas_calc["operacoes"]:
        print(f"  {op['passo']}: {op['formula']} = {op['resultado']}")

print("\n=== 4) metricas_flip_tool em paralelo (dados brutos, sem validacao) ===")
saida_metricas = metricas_flip_tool.invoke(
    {"dados_imovel_json": json.dumps(payload_tools["dados_imovel"], ensure_ascii=False, default=str)}
)
print(saida_metricas[:600])

print("\n=== artefatos no estado ===")
print("imagem:", json.dumps(
    ESTADO_AGENTE["imagem"].resumo() if ESTADO_AGENTE.get("imagem") else None,
    ensure_ascii=False,
))


fotos do imovel: 33

=== 1) classificar_fotos_tool (sem LLM) ===


11/23/2026 09:33:33 AM classificador: 33 fotos no anuncio (sem limite); o custo cresce por foto
11/23/2026 09:34:02 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:34:02 AM HTTP Request: HEAD https://huggingface.co/facebook/convnext-tiny-224/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
11/23/2026 09:34:02 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 180/180 [00:00<00:00, 13137.99it/s]
11/23/2026 09:34:03 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224/commits/main "HTTP/1.1 200 OK"
11/23/2026 09:34:03 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224/discussions?p=0 "HTTP/1.1 200 OK"
11/23/2026 09:34:03 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224/commits/refs%2Fpr%2F4 "HTTP/1.1 200 OK"
11/23/2026 09:34:03 AM HTTP Request

{"conservacao": "desgastado", "nota": 4.9, "confianca": 0.439, "interna": {"nota": 5.0, "conservacao": "desgaste leve", "fotos": 10}, "externa": {"nota": 4.8, "conservacao": "desgastado", "fotos": 23}, "proximo_passo": "validar_dados_tool", "fonte": "preditor"}

=== 1b) classificar_fotos_tool repetida (deve vir do cache) ===
{"conservacao": "desgastado", "nota": 4.9, "confianca": 0.439, "interna": {"nota": 5.0, "conservacao": "desgaste leve", "fotos": 10}, "externa": {"nota": 4.8, "conservacao": "desgastado", "fotos": 23}, "fonte": "cache", "proximo_passo": "validar_dados_tool", "aviso": "classificar_fotos_tool ja rodou

=== 2) validar_dados_tool (usa LLM; corrige metragem/valor) ===


11/23/2026 09:34:29 AM modelos_para(texto, Groq) via catálogo -> ['openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'qwen/qwen3.8-27b']
11/23/2026 09:34:29 AM modelos_para(texto, Cerebras) via catálogo -> ['gpt-oss-120b', 'qwen-3.8-27b']
11/23/2026 09:34:29 AM modelos_para(texto, API_Nvidia) via catálogo -> ['nvidia/nemotron-3-nano-omni-30b-a3b-reasoning', 'openai/gpt-oss-20b', 'z-ai/glm-5.3-flash']
11/23/2026 09:34:29 AM modelos_para(texto, Langchain_nvidia) via catálogo -> ['nvidia/nemotron-3-nano-omni-30b-a3b-reasoning', 'openai/gpt-oss-20b', 'z-ai/glm-5.3-flash']
11/23/2026 09:34:29 AM modelos_para(texto, Openai_nvidia) via catálogo -> ['nvidia/nemotron-3-nano-omni-30b-a3b-reasoning', 'openai/gpt-oss-20b', 'z-ai/glm-5.3-flash']
11/23/2026 09:34:29 AM modelos_para(texto, Zai) via catálogo -> ['glm-5.2', 'glm-5.3-flash', 'glm-5.3']
11/23/2026 09:34:29 AM 🔄 Rotando para provedor API_Nvidia
11/23/2026 09:34:29 AM Tentando API_Nvidia nvidia/nemotron-3-nano-omni-30b-a3b-reasoning
11/23/2026 0

{"dados_corrigidos": {"metragem": 46.0, "banheiros": 1, "vagas": 1, "quartos": 2, "valor_imovel": 330000.0, "bairro": "pirabeiraba", "tipo_imovel": "apartamento"}, "dados_consistentes": true, "inconsistencias_encontradas": [], "confianca_validacao": 0.99, "observacoes": "Os dados de metragem, quartos, banheiros, vagas, valor e tipo imóvel coincidem com a descrição. O bairro não é mencionado na descrição, portanto manteve-se o valor original. Não foram identificados erros ou contradições.", "possui_erros": "False", "metragem_corrigida": 46.0, "vagas_corrigidas": 1, "quartos_corrigidos": 2, "banheiros_corrigidos": null, "valor_imovel_corrigido": 330000.0, "tipo_imovel_corrigido": "apartamento", "bairro_corrigido": "pirabeiraba", "proximo_passo": "calcular_metricas_tool", "fonte": "validador"}

=== 3) calcular_metricas_tool (sem LLM, usa os dados validados) ===
{"preco_por_m2": 7173.91, "valor_m2_predicao": 7215.06, "arv_estimado": 326328.1, "custo_reforma_estimado": 69000.0, "lucro_proje

## Os 4 loops do artigo neste notebook

| Loop | Artigo | Aqui |
|---|---|---|
| 1. Agent | `create_agent` + tools | `create_deep_agent` "flip-investidor": `think_tool` + `registrar_decisao` no supervisor; subagentes `fotografo` (preditor ConvNeXt) e `dados` (métricas + validação) |
| 2. Verification | grader + retry com feedback | `VerificadorGraders` no `after_agent` (G1–G3) e o runner re-invoca a thread com o erro, 1 tentativa |
| 3. Event-driven | evento dispara o agente | `FILA_EVENTOS` + `processar_eventos(lote=N)` → `dados/agente_avaliacoes/runs.parquet` |
| 4. Hill climbing | análise melhora o harness | `c0de7006` mede graders e concordância com os rótulos humanos → `proposta_harness.md` |

Aprovacao humana: `interrupt_on={"registrar_decisao": True}`. `decisao_humana=None` pausa e devolve a
solicitacao; `retomar_avaliacao(thread, "approve")` continua. `RubricMiddleware` não foi usado porque exige
um modelo (LLM-as-judge); aqui o grader é determinístico.

In [11]:
# Aprovacao humana: para quando o agente pede para gravar a decisao.
r = await iniciar_avaliacao(CASOS["imovel_10"], "hitl-10", decisao_humana=None)

print("estado:", r["estado"])
if r["estado"] == "aguardando_humano":
    for i, req in enumerate(r["sessao"]["solicitacoes"], 1):
        print("\n--- solicitacao", i, "---")
        print(json.dumps(req, ensure_ascii=False, indent=2, default=str)[:1000])
    print("\nretome com retomar_avaliacao('hitl-10', DECISAO_HUMANA)")
else:
    print("concluido sem pausa (modelo nao chegou a pedir aprovacao)")
    print(r["saida"].model_dump_json(indent=2)[:600])

11/23/2026 09:34:58 AM HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
11/23/2026 09:34:58 AM classificador: 33 fotos no anuncio (sem limite); o custo cresce por foto
11/23/2026 09:34:59 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:35:00 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:35:00 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:35:00 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:35:00 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:35:01 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:35:01 AM HTTP Request: GET https://huggingface.co/api/mod

estado: aguardando_humano

--- solicitacao 1 ---
{
  "action_requests": [
    {
      "name": "registrar_decisao",
      "args": {
        "potencial_house_flip": "False",
        "score_potencial_flip": 1,
        "resumo": "O imóvel não atende ao critério mínimo de ROI de 15% (ROI real: -31.91%), resultando em potencial de flip 'False'. O preço máximo para a regra era R$ 195.387,21, muito abaixo do valor de anúncio de R$ 330.000,00."
      },
      "description": "Tool execution requires approval\n\nTool: registrar_decisao\nArgs: {'potencial_house_flip': 'False', 'score_potencial_flip': 1, 'resumo': \"O imóvel não atende ao critério mínimo de ROI de 15% (ROI real: -31.91%), resultando em potencial de flip 'False'. O preço máximo para a regra era R$ 195.387,21, muito abaixo do valor de anúncio de R$ 330.000,00.\"}"
    }
  ],
  "review_configs": [
    {
      "action_name": "registrar_decisao",
      "allowed_decisions": [
        "approve",
        "edit",
        "reject",
        "

In [12]:
# Sua decisao: "approve", "edit" ou "reject".
DECISAO_HUMANA = "reject"

if "hitl-10" not in SESSOES:
    print("rode a celula anterior primeiro")
else:
    r2 = await retomar_avaliacao("hitl-10", DECISAO_HUMANA)
    print("estado:", r2["estado"])
    if "saida" in r2:
        s = r2["saida"]
        print(f"ambiente={s.ambiente_fotos} conservacao={s.conservacao_fotos} nota={s.nota_conservacao} "
              f"flip={s.potencial_house_flip} score={s.score_potencial_flip} "
              f"tentativas={s.tentativas} aprovado={s.aprovado_graders}")
        print("decisao gravada:", s.graders["decisao"])
        print("\npensamentos:")
        for p in s.graders["pensamentos"]:
            print(f"- [{p['proxima_acao']}] {p['pensamento'][:150]}")

11/23/2026 09:36:05 AM THINK[registrar_decisao] Analisei as fotos (conservação desgastada), validei os dados (consistentes) e calculei as métricas. O resultado chave é que o ROI conservador é de -31.91% e, consequentemente, a 'passa_regra_15pct' é False. O potencial de flip é negado pela regra de 15%, independentemente da aparência das fotos.
11/23/2026 09:36:05 AM menu enxuto: ['calcular_metricas_tool', 'classificar_fotos_tool', 'validar_dados_tool'] ja rodaram, saem do menu
11/23/2026 09:36:08 AM HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
11/23/2026 09:36:08 AM decisao sintetizada do veredito estruturado


estado: concluido
ambiente=externa conservacao=desgastado nota=4.9 flip=False score=1.0 tentativas=1 aprovado=True
decisao gravada: {'potencial_house_flip': 'False', 'score_potencial_flip': 1.0, 'resumo': 'O cálculo financeiro resultou em um ROI conservador negativo (-31.91%), falhando em cumprir a regra mínima obrigatória de 15% para a aprovação do flip.', 'origem': 'veredito_estruturado'}

pensamentos:
- [validar_dados_tool] Classifiquei as fotos, identificando o ambiente interno com conservação de 'desgaste leve' e o ambiente externo como 'desgastado'. O próximo passo obr
- [registrar_decisao] Analisei as fotos (conservação desgastada), validei os dados (consistentes) e calculei as métricas. O resultado chave é que o ROI conservador é de -31


In [13]:
# Loop 3 (event-driven): eventos = fotos ROTULADAS -> agente -> runs.parquet.
from datetime import datetime, timezone

ARQ_RUNS = REPO / "dados" / "agente_avaliacoes" / "runs.parquet"
FILA_EVENTOS = []
FALHAS_RUNS = []  # runs sem veredito: falha de execucao, nao julgamento


def _n_fotos_ok(f):
    try:
        return len(f) >= 1
    except TypeError:
        return False


def gerar_eventos_rotulados(n=2, offset=0):
    """Um evento por foto rotulada, usando os dados do anuncio correspondente.

    Guarda `label_humano`/`ambiente_humano` no evento para o Loop 4 conseguir
    medir a concordancia do preditor sem depender de join por URL.
    """
    rot = pd.read_csv(REPO / "dados" / "rotulos_conservacao.csv")
    rot = rot[rot["label"].isin(LABELS)].drop_duplicates("url_foto")
    base = df[df["url"].isin(rot["url_anuncio"]) & df["fotos"].apply(_n_fotos_ok)]
    chosen = rot.iloc[offset:offset + n]
    for _, r in chosen.iterrows():
        linha = base[base["url"] == r["url_anuncio"]].iloc[0]
        FILA_EVENTOS.append({
            "evento": "foto_rotulada",
            "id": f"rot-{r['url_foto'][-24:]}",
            "payload": {
                "fotos_urls": [r["url_foto"]],
                "dados_imovel": converter_numpy({
                    "metragem": linha.get("metragem", 0),
                    "banheiros": linha.get("banheiros", 0),
                    "vagas": linha.get("vagas", 0),
                    "quartos": linha.get("quartos", 0),
                    "valor_imovel": linha.get("valor_imovel", 0),
                    "bairro": str(linha.get("bairro", "") or ""),
                    "tipo_imovel": str(linha.get("tipo_imovel", "") or ""),
                    "valor_predito": linha.get("valor_predito", 0),
                    "p50_bairro": linha.get("p50_bairro", 0),
                    "preco_por_m2": linha.get("preco_por_m2", 0),
                }),
                "descricao_texto": str(linha.get("descricao", "") or ""),
            },
            "label_humano": r["label"],
            "ambiente_humano": r["ambiente"],
            "gerado_em": datetime.now(timezone.utc).isoformat(),
        })
    return len(FILA_EVENTOS)


async def processar_eventos(lote=1, aprovacao_automatica=True, max_tentativas=1):
    """Consome ate `lote` eventos. Em producao use aprovacao_automatica=False.

    So entra no runs.parquet quem produziu veredito. Run sem decisao e falha de
    execucao (provedor caiu, tool pulada), nao "flip reprovado": misturar os
    dois media 3 runs de score 0 dentro das taxas do Loop 4. As falhas vao para
    FALHAS_RUNS e continuam visiveis aqui.
    """
    linhas = []
    gravaveis = []
    for _ in range(min(lote, len(FILA_EVENTOS))):
        ev = FILA_EVENTOS.pop(0)
        t0 = datetime.now(timezone.utc)
        r = await iniciar_avaliacao(
            ev["payload"], ev["id"],
            decisao_humana="approve" if aprovacao_automatica else None,
            max_tentativas=max_tentativas,
        )
        linha = {
            "evento_id": ev["id"], "evento": ev["evento"],
            "executado_em": t0.isoformat(), "estado": r["estado"],
            "url_foto": (ev["payload"]["fotos_urls"] or [""])[0],
            "bairro": str(ev["payload"]["dados_imovel"].get("bairro", "")),
            "label_humano": ev.get("label_humano"),
            "ambiente_humano": ev.get("ambiente_humano"),
        }
        if "saida" in r:
            s = r["saida"]
            linha.update({
                "ambiente": s.ambiente_fotos, "conservacao": s.conservacao_fotos,
                "nota": s.nota_conservacao, "flip": s.potencial_house_flip,
                "score": s.score_potencial_flip, "tentativas": s.tentativas,
                "aprovado_graders": s.aprovado_graders,
                "G1": s.graders["passou"]["G1"], "G2": s.graders["passou"]["G2"],
                "G3": s.graders["passou"]["G3"],
                "pensamentos": len(s.graders["pensamentos"]),
                "decisao_humana": s.graders.get("decisao_humana"),
                "validou": s.graders.get("validou"),
                "base_metricas": s.graders.get("metricas_base"),
                "motivo": (s.graders.get("erro_etapa") or s.graders.get("provedor_caiu") or ""),
            })
        linhas.append(linha)

        if "saida" in r and r["saida"].graders.get("decisao") is not None:
            gravaveis.append(linha)
        else:
            FALHAS_RUNS.append({
                "evento_id": ev["id"],
                "executado_em": linha["executado_em"],
                "motivo": linha.get("motivo") or r.get("estado", "sem saida"),
            })

    if gravaveis:
        ARQ_RUNS.parent.mkdir(parents=True, exist_ok=True)
        acumulado = pd.DataFrame(gravaveis)
        if ARQ_RUNS.exists():
            acumulado = pd.concat([pd.read_parquet(ARQ_RUNS), acumulado], ignore_index=True)
        acumulado.to_parquet(ARQ_RUNS, index=False)
        print(f"{ARQ_RUNS.name}: {len(acumulado)} execucoes acumuladas")
    print(f"deste lote: {len(gravaveis)} gravados, {len(FALHAS_RUNS)} recusados sem veredito (acumulado)")
    if FALHAS_RUNS:
        print(pd.DataFrame(FALHAS_RUNS[-5:]).to_string(index=False))
    return pd.DataFrame(linhas)


print("eventos criados:", gerar_eventos_rotulados(n=2))
print("fila:", [e["id"] for e in FILA_EVENTOS])

LOTE_EVENTOS = 2
eventos_processados = await processar_eventos(lote=LOTE_EVENTOS)
cols = ["evento_id", "estado", "bairro", "label_humano", "conservacao",
        "ambiente_humano", "ambiente", "nota", "flip", "score",
        "tentativas", "aprovado_graders", "pensamentos"]
print(eventos_processados.reindex(columns=cols).to_string(index=False))

eventos criados: 2
fila: ['rot-ension=870x707&seo=false', 'rot--quartos-6a9c3d0e-13.jpg']


11/23/2026 09:36:08 AM HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
11/23/2026 09:36:09 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:36:09 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:36:09 AM menu enxuto: ['classificar_fotos_tool'] ja rodaram, saem do menu
11/23/2026 09:36:12 AM HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
11/23/2026 09:36:12 AM Iniciando roteamento LLM
11/23/2026 09:36:12 AM modelos_para(texto, Groq) via catálogo -> ['openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'qwen/qwen3.8-27b']
11/23/2026 09:36:12 AM modelos_para(texto, Cerebras) via catálogo -> ['gpt-oss-120b', 'qwen-3.8-27b']
11/23/2026 09:36:12 AM modelos_para(texto, API_Nvidia) via catálogo -> ['nvidia/nemotron-3-nano-omni-30b-a3b-reasoning', 'openai/gpt-oss-20b', 'z-ai/glm-5.3-flash']


runs.parquet: 17 execucoes acumuladas
deste lote: 2 gravados, 0 recusados sem veredito (acumulado)
                   evento_id    estado          bairro   label_humano   conservacao ambiente_humano ambiente  nota  flip  score  tentativas  aprovado_graders  pensamentos
rot-ension=870x707&seo=false concluido anita garibaldi  desgaste leve desgaste leve         externa  externa   6.5 False    5.0           1              True            0
rot--quartos-6a9c3d0e-13.jpg concluido      atiradores bem conservado desgaste leve         externa  interna   7.3 False    3.0           1              True            0


In [14]:
# Loop 4 (hill climbing): execucoes acumuladas -> proposta de ajuste do harness.
from datetime import datetime, timezone

ARQ_PROPOSTA = REPO / "dados" / "agente_avaliacoes" / "proposta_harness.md"

runs = pd.read_parquet(ARQ_RUNS) if ARQ_RUNS.exists() else pd.DataFrame()

# Coluna de conservacao unificada NA LEITURA (o arquivo em disco nao e reescrito):
# runs antigos gravaram `desgaste`, os novos gravam `conservacao`.
if len(runs) and "desgaste" in runs.columns:
    if "conservacao" in runs.columns:
        runs["conservacao"] = runs["conservacao"].fillna(runs["desgaste"])
    else:
        runs["conservacao"] = runs["desgaste"]
    runs = runs.drop(columns=["desgaste"])

# Taxas de veredito sobre quem de fato julgou: score 0 de run interrompido nao
# e flip reprovado.
com_veredito = runs[runs["flip"].notna()] if len(runs) and "flip" in runs.columns else runs
print("runs carregados:", len(runs), "| com veredito:", len(com_veredito))


def _concordancia(runs):
    """Compara desgaste/ambiente do preditor com o rotulo humano do evento."""
    if not len(runs) or "label_humano" not in runs.columns:
        return None, "sem eventos rotulados; rode gerar_eventos_rotulados() no Loop 3"
    _COL = "conservacao" if "conservacao" in runs.columns else "desgaste"
    r = runs.dropna(subset=["label_humano"])
    if not len(r):
        return None, "nenhum run com label_humano"
    conc = {
        "n": len(r),
        "acuracia_conservacao": round(float((r[_COL] == r["label_humano"]).mean()), 3),
        "acuracia_ambiente": round(float(
            (r["ambiente"] == r["ambiente_humano"].replace({"ambigua": "desconhecido"})).mean()
        ), 3) if "ambiente_humano" in r.columns else None,
    }
    matriz = pd.crosstab(r["label_humano"], r[_COL])
    return conc, matriz


concordancia, matriz_or_aviso = _concordancia(runs)
if concordancia:
    print("\nconcordancia do preditor com o rotulo humano:")
    print(json.dumps(concordancia, indent=2, default=str))
    print(matriz_or_aviso.to_string())
    matriz = matriz_or_aviso
    aviso = None
else:
    aviso = matriz_or_aviso
    matriz = None
    print("\naviso:", aviso)


def _taxa(frame, col):
    if not len(frame) or col not in frame.columns:
        return None
    return round(float((~frame[col].astype(bool)).mean()), 3)


resumo = {
    "total_execucoes": int(len(runs)),
    "n_util": int(len(com_veredito)),
    "concluidas": int((com_veredito["estado"] == "concluido").sum()) if len(com_veredito) else 0,
    "aguardando_humano": int((runs["estado"] == "aguardando_humano").sum()) if len(runs) else 0,
    "aprovadas_graders": round(float(com_veredito["aprovado_graders"].mean()), 3) if len(com_veredito) else None,
    "reprovado_G1": _taxa(com_veredito, "G1"),
    "reprovado_G2": _taxa(com_veredito, "G2"),
    "reprovado_G3": _taxa(com_veredito, "G3"),
    "tentativas_medias": round(float(com_veredito["tentativas"].mean()), 2) if len(com_veredito) else None,
    "flip_aprovados": round(float((com_veredito["flip"] == "True").mean()), 3) if len(com_veredito) else None,
}
print("\nsinais:", json.dumps(resumo, indent=2, default=str))

propostas = []
if len(runs):
    if resumo["n_util"] < resumo["total_execucoes"]:
        propostas.append(
            f"{resumo['total_execucoes'] - resumo['n_util']} runs historicos nao tinham veredito "
            "(score 0 de execucao interrompida). Ja foram excluidos das taxas: olhar FALHAS_RUNS "
            "e as colunas `validou`/`motivo` antes de culpar o prompt."
        )
    if resumo["reprovado_G2"]:
        propostas.append(
            "G2 reprova " + f"{resumo['reprovado_G2']:.0%} dos runs: as faixas FAIXA_NOTA "
            "nao alinharam nota e classe. Recalibrar com as fotos rotuladas."
        )
    if resumo["reprovado_G3"]:
        propostas.append(
            "G3 reprova " + f"{resumo['reprovado_G3']:.0%}: o veredito saiu sem score/justificativa validos; "
            "reforcar no prompt que so use valores devolvidos pelas ferramentas."
        )
    if resumo["tentativas_medias"] and resumo["tentativas_medias"] > 1:
        propostas.append(
            "Media de tentativas " + f"{resumo['tentativas_medias']}: aumentar max_tentativas ou revisar o grader."
        )
    if resumo["aguardando_humano"]:
        propostas.append(
            f"{resumo['aguardando_humano']} runs pararam para aprovacao humana: definir fila/SLA de revisao."
        )
    if concordancia and concordancia["acuracia_conservacao"] < 0.6:
        propostas.append(
            "Conservacao do preditor bate com o humano em so "
            + f"{concordancia['acuracia_conservacao']:.0%}: retreinar o classificador com mais fotos "
            "das classes raras (bem conservado, precario) antes de confiar no veredito."
        )
    if concordancia and concordancia.get("acuracia_ambiente") is not None and concordancia["acuracia_ambiente"] < 0.9:
        propostas.append(
            "Ambiente com concordancia de "
            + f"{concordancia['acuracia_ambiente']:.0%}: rever as fotos ambiguas (varanda, area comum)."
        )
if not propostas:
    propostas.append("Nenhum ajuste de harness indicado pelos sinais atuais.")

texto = [
    "# Proposta de ajuste do harness",
    "",
    "Gerada em " + datetime.now(timezone.utc).isoformat() + " pelo Loop 4 do notebook react_agent_valiador_2.ipynb.",
    "",
    "## Sinais",
    "```json", json.dumps(resumo, indent=2, default=str), "```",
    "",
]
if concordancia:
    texto += ["## Concordancia com rotulos humanos", "```json", json.dumps(concordancia, indent=2, default=str), "```",
              "", "```", matriz.to_string(), "```", ""]
if aviso:
    texto += ["## Aviso", aviso, ""]
texto += ["## Propostas", ""] + ["- " + p for p in propostas]

ARQ_PROPOSTA.parent.mkdir(parents=True, exist_ok=True)
ARQ_PROPOSTA.write_text("\n".join(texto), encoding="utf-8")
print("\nproposta salva em:", ARQ_PROPOSTA)
print("\n".join(texto[-12:]))

runs carregados: 17 | com veredito: 17

concordancia do preditor com o rotulo humano:
{
  "n": 17,
  "acuracia_conservacao": 0.529,
  "acuracia_ambiente": 0.529
}
conservacao     desgaste leve
label_humano                 
bem conservado              8
desgaste leve               9

sinais: {
  "total_execucoes": 17,
  "n_util": 17,
  "concluidas": 17,
  "aguardando_humano": 0,
  "aprovadas_graders": 0.765,
  "reprovado_G1": 0.118,
  "reprovado_G2": 0.118,
  "reprovado_G3": 0.235,
  "tentativas_medias": 1.12,
  "flip_aprovados": 0.0
}

proposta salva em: c:\Users\jefer\Documents\Ciencia-de-dados\Preco-Imoveis\dados\agente_avaliacoes\proposta_harness.md

```
conservacao     desgaste leve
label_humano                 
bem conservado              8
desgaste leve               9
```

## Propostas

- G2 reprova 12% dos runs: as faixas FAIXA_NOTA nao alinharam nota e classe. Recalibrar com as fotos rotuladas.
- G3 reprova 24%: o veredito saiu sem score/justificativa validos; reforcar no prom

## Lote Atiradores — 10 apartamentos por preço/m², com LangSmith

O agente avalia 10 imóveis reais do **Atiradores** (`apartamento`), ordenados do menor para o maior
`preco_por_m2`, com **uma trace no LangSmith por avaliação** e o veredito gravado como *feedback* na
trace — sem MLflow.

A célula do lote avalia **todos os 10** por padrão (`N_LOTE` da célula de seleção). Para conferir
só o primeiro, use `FLIP_LOTE_PILOTO=1` no ambiente antes de rodar a célula.

Dois cuidados com os dados, visíveis na célula de seleção:

- **Filtro de sanidade** — na ordenação bruta os 4 primeiros são `valor_imovel = 0` (2×) e
  `valor_imovel = 8.000` (R$ 36/m², 2×): locação e anúncio sem preço misturados na base de venda.
- **Dedup** — o mesmo apartamento aparece em `chave_mao` e `zap_imoveis` com `valor_predito`
  diferente. Sem dedup, 10 linhas seriam ~7 imóveis avaliados duas vezes.

Tracing: o `.env` da raiz já traz `LANGCHAIN_TRACING_V2=true` e `LANGCHAIN_API_KEY`, então o
`create_agent` traceia LLM, tools e middleware sem instrumentação. A seção só troca o projeto
(`FLIP_LS_PROJETO`, default `Agente-Imovel-Atiradores`) e marca cada trace com `imovel:<id>` e
`lote:<tag>` para achar no UI.

**Um imóvel não é uma trace**: cada `ainvoke` (turno inicial, retry dos graders, recuperação) abre uma
raiz própria, e um run-filho também herda a metadata do imóvel. A auditoria agrega por imóvel e aponta a
raiz **mais recente** — é nela que o feedback é gravado.


In [15]:
# Selecao do lote: 10 apartamentos do Atiradores, do menor para o maior preco_por_m2.
import math

from agente_potencial_flip.tools_flip import calcular_metricas_flip

BAIRRO_LOTE = "atiradores"
TIPO_LOTE = "apartamento"
N_LOTE = 10

lote_bruto = df[(df["bairro"] == BAIRRO_LOTE) & (df["tipo_imovel"] == TIPO_LOTE)].copy()
lote_bruto["ppm2"] = pd.to_numeric(lote_bruto["preco_por_m2"], errors="coerce")
lote_bruto["valor"] = pd.to_numeric(lote_bruto["valor_imovel"], errors="coerce")
lote_bruto["predito"] = pd.to_numeric(lote_bruto["valor_predito"], errors="coerce")
lote_bruto["metr"] = pd.to_numeric(lote_bruto["metragem"], errors="coerce")
lote_bruto["n_fotos"] = lote_bruto["fotos"].apply(_n_fotos)

# Sanidade: a ordenacao pura comeca em R$ 0 e R$ 8.000. O agente aprovaria um flip
# comprado por 8 reais, e isso nao e um resultado - e dado sujo.
lote_sano = lote_bruto[
    (lote_bruto["valor"] > 0)
    & lote_bruto["ppm2"].apply(lambda v: bool(v) and math.isfinite(float(v)))
    & (lote_bruto["ppm2"] >= 1500)
    & (lote_bruto["predito"] > 0)
    & (lote_bruto["metr"] >= 30)
    & (lote_bruto["n_fotos"] >= 3)
].copy()

# Dedup: mesmo (metragem, valor, quartos) vindo de duas fontes -> fica o mais barato.
lote_sano["_chave"] = (
    lote_sano["metr"].round(1).astype(str) + "|"
    + lote_sano["valor"].round(0).astype(str) + "|"
    + lote_sano["quartos"].astype(str)
)
DESCARTADOS = lote_bruto.loc[~lote_bruto.index.isin(lote_sano.index)].sort_values("ppm2")
SELECAO_ATIRADORES = (
    lote_sano.sort_values("ppm2")
    .drop_duplicates("_chave", keep="first")
    .head(N_LOTE)
    .reset_index(drop=True)
)

print(f"{BAIRRO_LOTE} + {TIPO_LOTE}: {len(lote_bruto)} linhas | {len(lote_sano)} sas | "
      f"{len(SELECAO_ATIRADORES)} no lote")

print("\nmais baratos DESCARTADOS (conferir se era oportunidade real):")
for _, r in DESCARTADOS.head(4).iterrows():
    print(f"  ppm2 {r['ppm2']:9.1f} | valor {r['valor']:>10.0f} | metragem {r['metr']:7.1f} "
          f"| fotos {r['n_fotos']} | fonte {r['fonte']}")

print(f"\no lote, do menor para o maior preco_por_m2:")
print(SELECAO_ATIRADORES[["ppm2", "valor", "metr", "quartos", "banheiros", "vagas",
                          "n_fotos", "predito", "p50_bairro", "fonte"]].to_string(index=False))

# Previa deterministica: a regua que o agente tem que confirmar (ou contestar).
PREVIA = []
for i, r in SELECAO_ATIRADORES.iterrows():
    m = calcular_metricas_flip({
        "valor_imovel": float(r["valor"]), "metragem": float(r["metr"]),
        "valor_predito": float(r["predito"]), "p50_bairro": float(r["p50_bairro"]),
    })
    PREVIA.append({
        "imovel": f"atiradores-{i + 1:02d}", "ppm2": round(float(r["ppm2"])),
        "arv_previsto": m["arv_estimado"], "teto_previsto": m["preco_maximo_para_regra"],
        "roi_previsto": m["roi_conservador"], "passa_previsto": m["passa_regra_15pct"],
    })
PREVIA = pd.DataFrame(PREVIA)
print("\nprevia da conta (sem LLM):")
print(PREVIA.to_string(index=False))

PAYLOADS_ATIRADORES = []
for i, r in SELECAO_ATIRADORES.iterrows():
    p = montar_payload(r)
    p["_id"] = f"atiradores-{i + 1:02d}"
    p["_idx"] = int(i)
    PAYLOADS_ATIRADORES.append(p)
print("\npayloads:", len(PAYLOADS_ATIRADORES), "| fotos por imovel:",
      [len(p["fotos_urls"]) for p in PAYLOADS_ATIRADORES])

atiradores + apartamento: 1073 linhas | 1056 sas | 10 no lote

mais baratos DESCARTADOS (conferir se era oportunidade real):
  ppm2       0.0 | valor          0 | metragem   123.0 | fotos 25 | fonte chave_mao
  ppm2       0.0 | valor          0 | metragem   545.0 | fotos 18 | fonte chave_mao
  ppm2      36.4 | valor       8000 | metragem   220.0 | fotos 26 | fonte chave_mao
  ppm2      37.4 | valor       8000 | metragem   214.0 | fotos 16 | fonte chave_mao

o lote, do menor para o maior preco_por_m2:
       ppm2     valor  metr  quartos  banheiros  vagas  n_fotos      predito   p50_bairro       fonte
2962.241462  615850.0 207.9        2          2      2       20 7.230896e+05 10089.285714   chave_mao
3178.571429  845500.0 266.0        2          3      2       17 8.864928e+05 10089.285714 zap_imoveis
3499.147727  615850.0 176.0        2          2      2       20 9.406782e+05 10089.285714 zap_imoveis
3846.153846  450000.0 117.0        3          2      0       14 5.248667e+05 10089.285

In [16]:
PAYLOADS_ATIRADORES

[{'fotos_urls': ['https://www.chavesnamao.com.br/imn/1200x0800/N/75/imoveis/130266/47067536/sc-joinville-atiradores-rua-xavantes-cobertura-a-venda-2-quartos-6ab6633b-1.jpg',
   'https://www.chavesnamao.com.br/imn/1200x0800/N/75/imoveis/130266/47067536/sc-joinville-atiradores-rua-xavantes-cobertura-a-venda-2-quartos-6ab6633b-2.jpg',
   'https://www.chavesnamao.com.br/imn/1200x0800/N/75/imoveis/130266/47067536/sc-joinville-atiradores-rua-xavantes-cobertura-a-venda-2-quartos-6ab6633b-3.jpg',
   'https://www.chavesnamao.com.br/imn/1200x0800/N/75/imoveis/130266/47067536/sc-joinville-atiradores-rua-xavantes-cobertura-a-venda-2-quartos-6ab6633b-4.jpg',
   'https://www.chavesnamao.com.br/imn/1200x0800/N/75/imoveis/130266/47067536/sc-joinville-atiradores-rua-xavantes-cobertura-a-venda-2-quartos-6ab6633b-5.jpg',
   'https://www.chavesnamao.com.br/imn/1200x0800/N/75/imoveis/130266/47067536/sc-joinville-atiradores-rua-xavantes-cobertura-a-venda-2-quartos-6ab6633b-6.jpg',
   'https://www.chavesnama

In [17]:
# LangSmith: projeto do lote. O .env da raiz ja traz LANGCHAIN_TRACING_V2=true e a
# chave, entao o create_agent traceia LLM, tools e middleware sem instrumentacao.
import os

from langsmith import Client

LS_PROJETO = os.environ.get("FLIP_LS_PROJETO", "Agente-Imovel-Atiradores")
os.environ["LANGCHAIN_PROJECT"] = LS_PROJETO
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")

ls = Client()
print("LangSmith:", os.environ.get("LANGCHAIN_ENDPOINT") or "(nao configurado)")
print("projeto:", LS_PROJETO, "| chave:", "ok" if os.getenv("LANGCHAIN_API_KEY") else "AUSENTE")
print("UI: https://smith.langchain.com  (filtrar tag lote:"
      + os.environ.get("FLIP_LOTE_TAG", "atiradores") + ")")

LangSmith: https://api.smith.langchain.com
projeto: Agente-Imovel-Atiradores | chave: ok
UI: https://smith.langchain.com  (filtrar tag lote:atiradores)


In [18]:
# Lote: o agente avalia os 10; veredito gravado como feedback no LangSmith.
import json
import time
from datetime import datetime, timedelta, timezone

# Quantos imoveis avaliar. O default e o lote INTEIRO (o que a c0de8102 montou);
# antes o default era 1 e a celula avaliava so o primeiro sem avisar.
N_NO_LOTE = len(PAYLOADS_ATIRADORES)
LOTE_PILOTO = int(os.environ.get("FLIP_LOTE_PILOTO") or N_NO_LOTE)
if LOTE_PILOTO <= 0:  # 0 ou negativo = todos
    LOTE_PILOTO = N_NO_LOTE
LOTE_PILOTO = min(LOTE_PILOTO, N_NO_LOTE)  # nunca fatia mais do que existe
PAYLOADS_DO_LOTE = PAYLOADS_ATIRADORES[:LOTE_PILOTO]
print(f"lote: {len(PAYLOADS_DO_LOTE)} de {N_NO_LOTE} imoveis"
      " (FLIP_LOTE_PILOTO=1 para rodar so o piloto)")

# O bolt-e4b responde sem calcular em ~50% dos imoveis; com nemotron-3-ultra os
# mesmos imoveis completam. Troca so nesta secao e restaura no fim.
MODELO_LOTE = os.environ.get("FLIP_MODELO_LOTE", 
                             #"z-ai/glm-5.3-flash",
                             #"nvidia/nemotron-3.5-bolt-e4b",
                             #"minimaxai/minimax-m3.1-flash"
                             "nvidia/nemotron-3-ultra-550b-a55b"
                             )
MODELO_AGENTE_ANTERIOR = globals().get("MODELO_AGENTE")
FLIP_MODEL_ANTERIOR = os.environ.get("FLIP_MODEL", "")
if MODELO_LOTE:
    os.environ["FLIP_MODEL"] = MODELO_LOTE
    MODELO_AGENTE = criar_modelo_flip()
    print("modelo do lote:", MODELO_LOTE)


def _raiz_por_imovel(janela_min=180):
    """Uma varredura: imovel -> trace principal (a mais nova do imovel).

    Um imovel gera varias raizes (turno inicial, retry, recuperacao). O feedback
    vai na principal, que e a que carrega a analise completa.
    """
    desde = datetime.now(timezone.utc) - timedelta(minutes=janela_min)
    vistos, brutos = set(), []
    for r in ls.list_runs(project_name=LS_PROJETO, start_time=desde):
        if r.id in vistos:
            continue
        vistos.add(r.id)
        brutos.append(r)

    # So raizes: um filho (VerificadorGraders, ToolNode) tambem carrega a
    # metadata do imovel e, sendo mais novo, seria escolhido no lugar da raiz --
    # foi assim que o feedback caiu num run-filho e a auditoria nao o achou.
    brutos = [r for r in brutos if r.parent_run_id is None]

    nome_de = {}
    for r in brutos:
        meta = (r.extra or {}).get("metadata") or {}
        nome = str(meta.get("imovel") or "")
        if not nome:
            import re as _re

            m = _re.match(r"auto-(.+)-[0-9a-f]{10}$", str(meta.get("thread_id") or ""))
            nome = m.group(1) if m else ""
        nome_de[r.id] = nome

    filhos = {}
    for r in brutos:
        if r.parent_run_id:
            filhos.setdefault(r.parent_run_id, []).append(r)

    # A trace principal e a MAIS NOVA do imovel (a do lote atual). Ordenar por
    # numero de filhos pegava execucao antiga com mais repeticoes.
    ordem = {r.id: (r.start_time or datetime.min.replace(tzinfo=timezone.utc)) for r in brutos}
    principal = {}
    for r in brutos:
        nome = nome_de.get(r.id, "")
        if not nome:
            continue
        if nome not in principal or ordem[r.id] > ordem[principal[nome].id]:
            principal[nome] = r
    return principal


def _gravar_feedback(resultados):
    """Feedback na trace principal de cada imovel (scores + veredito em JSON)."""
    principal = _raiz_por_imovel()
    gravados = 0
    for r in resultados:
        traco = principal.get(r["imovel"])
        if traco is None:
            print(f"  (sem trace para {r['imovel']}: feedback nao gravado)")
            continue

        def fb(chave, valor, comentario=None, _t=traco):
            try:
                ls.create_feedback(str(_t.id), key=chave, score=float(valor),
                                   comment=comentario)
            except Exception as exc:  # noqa: BLE001
                print(f"  feedback {chave} falhou:", type(exc).__name__)

        # O score do feedback aceita so +-99999.9999: valores em R$ vao no comentario.
        fb("roi_conservador", r["roi"] or 0)
        fb("score_potencial_flip", r["score"] or 0)
        fb("aprovado_graders", 1 if r["aprovado"] else 0)
        fb("potencial_house_flip", 1 if str(r["flip"]) == "True" else 0,
           comentario=str(r["flip"]))
        fb("veredito", 1, comentario=json.dumps(r, ensure_ascii=False, default=str))

        r["link"] = getattr(traco, "app_path", None) or ""
        gravados += 1
        print(f"  {r['imovel']:16s} {r['link']}")
    return gravados


resultados_lote = []
for i, payload in enumerate(PAYLOADS_DO_LOTE, 1):
    linha = SELECAO_ATIRADORES.iloc[payload["_idx"]]
    t0 = time.perf_counter()
    saida = await avaliar_imovel_agente(payload, max_tentativas=2)
    metricas = saida.graders.get("metricas_tool") or {}
    dur = time.perf_counter() - t0

    resultados_lote.append({
        "imovel": payload["_id"], "ppm2": round(float(linha["ppm2"])),
        "fotos": len(payload["fotos_urls"]),
        "roi": metricas.get("roi_conservador"),
        "teto": metricas.get("preco_maximo_para_regra"),
        "arv": metricas.get("arv_estimado"),
        "flip": saida.potencial_house_flip, "score": saida.score_potencial_flip,
        "aprovado": saida.aprovado_graders,
        "base_dados": saida.graders.get("metricas_base"),
        "quem": saida.graders.get("quem_analisou"),
        "pulou": ",".join(saida.graders.get("etapas_runner") or []) or "-",
        "correcoes": json.dumps((metricas or {}).get("correcoes_aplicadas"), ensure_ascii=False),
        "dur_s": round(dur, 1),
    })
    print(f"{i:2d}/{len(PAYLOADS_DO_LOTE)} {payload['_id']} ppm2 {linha['ppm2']:7.0f} -> "
          f"{saida.potencial_house_flip} | score {saida.score_potencial_flip} "
          f"| roi {metricas.get('roi_conservador')} | aprov {saida.aprovado_graders} "
          f"| {dur:.0f}s", flush=True)

try:
    from langchain_core.tracers.langchain import wait_for_all_tracers

    wait_for_all_tracers()
except Exception:  # noqa: BLE001
    pass

if len(PAYLOADS_DO_LOTE) < N_NO_LOTE:
    print(f"\n(piloto: {len(PAYLOADS_DO_LOTE)} de {N_NO_LOTE} imoveis; "
          "tire FLIP_LOTE_PILOTO para rodar o lote completo)")

print("\nfeedback no LangSmith (trace principal de cada imovel):")
_gravar_feedback(resultados_lote)

tabela_lote = pd.DataFrame(resultados_lote)
print()
print(tabela_lote.drop(columns=["link"], errors="ignore").to_string(index=False))

if MODELO_LOTE and MODELO_AGENTE_ANTERIOR is not None:
    if FLIP_MODEL_ANTERIOR:
        os.environ["FLIP_MODEL"] = FLIP_MODEL_ANTERIOR
    else:
        os.environ.pop("FLIP_MODEL", None)
    MODELO_AGENTE = MODELO_AGENTE_ANTERIOR
    print("\nmodelo do agente restaurado para o default das outras secoes")

lote: 10 de 10 imoveis (FLIP_LOTE_PILOTO=1 para rodar so o piloto)
modelo do lote: nvidia/nemotron-3-ultra-550b-a55b


11/23/2026 09:39:55 AM HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
11/23/2026 09:39:55 AM THINK[task:fotografo] Inicio da avaliacao do imovel para flip. Tenho 20 fotos, dados do anuncio e descricao. Vou seguir a ordem obrigatoria: 1) classificar fotos, 2) validar dados, 3) calcular metricas, 4) think_tool conciliando, 5) registrar_decisao.
11/23/2026 09:39:55 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:39:56 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:39:56 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:39:56 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:39:56 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 0

 1/10 atiradores-01 ppm2    2962 -> True | score 8.0 | roi 0.3972 | aprov True | 230s


11/23/2026 09:43:40 AM HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
11/23/2026 09:43:40 AM Retrying request to /chat/completions in 0.429130 seconds
11/23/2026 09:43:41 AM HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
11/23/2026 09:43:41 AM chamada do agente falhou: RateLimitError: Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
11/23/2026 09:43:41 AM agente nao avaliou as fotos; usando o preditor direto
11/23/2026 09:43:42 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:43:43 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:43:43 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:43:43 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny

 2/10 atiradores-02 ppm2    3179 -> False | score 0.0 | roi None | aprov False | 11s


11/23/2026 09:43:52 AM HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
11/23/2026 09:43:52 AM Retrying request to /chat/completions in 0.387806 seconds
11/23/2026 09:44:25 AM HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
11/23/2026 09:44:25 AM THINK[classificar_fotos_tool] Iniciando avaliacao de cobertura no bairro Atiradores (Joinville) - 176m2, 2 quartos, 2 banheiros, 2 vagas, R$ 615.850. Tenho 20 fotos para classificar. Primeira etapa: classificar_fotos_tool para obter ambiente e conservacao.
11/23/2026 09:44:26 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:44:26 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:44:26 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:44:27 AM HTTP Request: G

... (13 outputs left out)

11/23/2026 09:55:30 AM HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
11/23/2026 09:55:30 AM THINK[task:fotografo] Iniciando avaliação de fix & flip para apartamento de 321m² no Bairro Atiradores, valor anunciado R$ 1.493.310. Vou seguir o processo obrigatório de 5 etapas. Primeira etapa: classificar as 36 fotos do anúncio com o preditor ConvNeXt para entender ambiente e conservação.
11/23/2026 09:55:30 AM classificador: 38 fotos no anuncio (sem limite); o custo cresce por foto
11/23/2026 09:55:31 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:55:32 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:55:32 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 200 OK"
11/23/2026 09:55:32 AM HTTP Request: GET https://huggingface.co/api/models/facebook/convnext-tiny-224 "HTTP/1.1 20

10/10 atiradores-10 ppm2    4652 -> False | score 0.0 | roi None | aprov False | 31s

feedback no LangSmith (trace principal de cada imovel):


C:\Users\jefer\AppData\Local\Temp\ipykernel_41840\1418744207.py:41: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  for r in ls.list_runs(project_name=LS_PROJETO, start_time=desde):


  (sem trace para atiradores-01: feedback nao gravado)
  (sem trace para atiradores-02: feedback nao gravado)
  (sem trace para atiradores-03: feedback nao gravado)
  (sem trace para atiradores-04: feedback nao gravado)
  (sem trace para atiradores-05: feedback nao gravado)
  (sem trace para atiradores-06: feedback nao gravado)
  (sem trace para atiradores-07: feedback nao gravado)
  (sem trace para atiradores-08: feedback nao gravado)
  (sem trace para atiradores-09: feedback nao gravado)
  (sem trace para atiradores-10: feedback nao gravado)

       imovel  ppm2  fotos    roi      teto        arv  flip  score  aprovado            base_dados   quem pulou                                               correcoes  dur_s
atiradores-01  2962     20 0.3972 748227.36 1249401.62  True    8.0      True              validado agente     - {"metragem": {"de": 207.89999999999998, "para": 176.0}}  229.6
atiradores-02  3179     17    NaN       NaN        NaN False    0.0     False                  No

In [19]:
tabela_lote

,imovel,ppm2,fotos,roi,teto,arv,flip,score,aprovado,base_dados,quem,pulou,correcoes,dur_s
0,atiradores-01,2962,20,0.3972,748227.36,1249401.62,True,8.0,True,validado,agente,-,"{""metragem"": {""de"": 207.89999999999998, ""para""...",229.6
1,atiradores-02,3179,17,NaN,NaN,NaN,False,0.0,False,None,agente,-,null,10.9
2,atiradores-03,3499,20,0.5562,833370.80,1358196.02,True,8.0,True,validado_sem_correcao,agente,-,{},126.3
3,atiradores-04,3846,14,NaN,NaN,NaN,False,0.0,False,None,agente,-,null,9.4
4,atiradores-05,4012,17,0.3399,757343.67,1237716.91,True,8.5,True,validado_sem_correcao,agente,-,{},116.3
5,atiradores-06,4211,36,0.3835,481212.30,773215.72,True,8.5,True,validado_sem_correcao,agente,-,{},112.1
6,atiradores-07,4455,18,0.1967,468265.91,766673.11,True,8.0,True,validado_sem_correcao,agente,-,{},88.9
7,atiradores-08,4537,23,NaN,NaN,NaN,False,0.0,False,None,agente,-,null,22.7
8,atiradores-09,4583,34,0.2256,586161.19,948983.74,True,7.5,True,validado_sem_correcao,agente,-,{},217.1
9,atiradores-10,4652,38,NaN,NaN,NaN,False,0.0,False,None,agente,-,null,31.3


## Experimento — ARV = média(mediana do bairro, predição do imóvel)

A célula do lote roda com a âncora de sempre: `arv = max(valor_m2_predicao, p50_bairro) × metragem`.
No Atiradores a mediana ganha em **10 de 10** imóveis e funciona como um prêmio de valorização
implícito de **+116%** sobre a predição (4.661/m² → 10.079/m²) — escondido numa constante de portal.

Este experimento injeta, **só neste notebook**, o desempate novo:

    ARV/m² = (valor_m2_predicao + p50_bairro) / 2

- **Nada em `src/`, no prompt ou nos graders muda.** A pergunta é como as **respostas** do agente
  ficam quando só a conta muda.
- Pré-conta sem LLM: 10/10 aprovados → **7/10**, ROI mediano +0,77 → **+0,29**.
- O run vai para um **projeto LangSmith próprio** (`Agente-Imovel-Atiradores-Blend`, tag `lote:blend`)
  para não misturar com o baseline — que já está em `Agente-Imovel-Atiradores` com feedback.
- A última célula compara os dois por feedback e põe as **justificativas** do agente lado a lado.
- Restauração: `_restaurar_ancora()` volta a conta original (roda sozinha no fim da célula do lote).


In [20]:
# EXPERIMENTO: ARV = media(p50_bairro, valor_m2_predicao). Reversivel: _restaurar_ancora().
import agente_potencial_flip.tools_flip as tf
from agente_potencial_flip.tools_flip import ROI_MINIMO

CALCULAR_METRICAS_ORIGINAL = tf.calcular_metricas_flip
FORMULAS_METRICAS_ORIGINAIS = dict(tf.FORMULAS_METRICAS)
ENTRADAS_METRICAS_ORIGINAIS = dict(tf.ENTRADAS_METRICAS)


def _fl(x):
    try:
        x = float(x)
    except (TypeError, ValueError):
        return 0.0
    return x if x == x and x not in (float("inf"), float("-inf")) else 0.0


def _calcular_metricas_blend(dados, custo_m2=None):
    """calcular_metricas_flip com ARV = media(mediana do bairro, predicao do imovel).

    Fonte unica preservada: chama a original e sobrescreve so o que depende do ARV.
    Sem predicao OU sem mediana do bairro, devolve a conta original (nao inventa ARV).
    """
    m = CALCULAR_METRICAS_ORIGINAL(dados, custo_m2)
    m2p = _fl(m.get("valor_m2_predicao"))
    p50 = _fl(m.get("valor_m2_bairro"))
    m2 = _fl(m.get("metragem_usada"))
    if not (m2p and p50 and m2):
        return m
    m2_blend = (m2p + p50) / 2
    arv = round(m2_blend * m2, 2)
    valor = _fl(m.get("valor_imovel"))
    custo_reforma = _fl(m.get("custo_reforma_estimado"))
    lucro = arv * 0.90 - valor - custo_reforma if arv and valor else 0.0
    roi = (lucro / valor) if valor else 0.0
    m.update(
        valor_m2_blend=round(m2_blend, 2),
        arv_fonte="blend_mediana_predicao",
        arv_estimado=arv,
        lucro_projetado=round(lucro, 2),
        roi_conservador=round(roi, 4),
        passa_regra_15pct=bool(roi >= ROI_MINIMO),
        preco_maximo_para_regra=round((arv * 0.90 - custo_reforma) / (1 + ROI_MINIMO), 2) if arv else 0.0,
        margem_vs_arv_pct=round((arv - valor) / arv * 100, 2) if arv and valor else 0.0,
    )
    return m


def _restaurar_ancora():
    """Volta a conta original (max(predicao, p50_bairro)) e as fomas em texto."""
    tf.calcular_metricas_flip = CALCULAR_METRICAS_ORIGINAL
    tf.FORMULAS_METRICAS.clear()
    tf.FORMULAS_METRICAS.update(FORMULAS_METRICAS_ORIGINAIS)
    tf.ENTRADAS_METRICAS.clear()
    tf.ENTRADAS_METRICAS.update(ENTRADAS_METRICAS_ORIGINAIS)
    print("ancora restaurada: arv = max(valor_m2_predicao, p50_bairro) * metragem")


tf.calcular_metricas_flip = _calcular_metricas_blend
tf.FORMULAS_METRICAS["arv_estimado"] = "(valor_m2_predicao + valor_m2_bairro) / 2 * metragem"
tf.ENTRADAS_METRICAS["arv_estimado"] = ("valor_m2_predicao", "valor_m2_bairro", "metragem")
print("patch ativo: ARV = media(mediana do bairro, predicao do imovel)")
print("  formula agora:", tf.FORMULAS_METRICAS["arv_estimado"])

# conferencia sintetica: mesma entrada, as duas ancoras lado a lado
_probe = {"valor_imovel": 480000, "metragem": 100, "valor_predito": 688000, "p50_bairro": 4800}
o = CALCULAR_METRICAS_ORIGINAL(dict(_probe))
n = _calcular_metricas_blend(dict(_probe))
assert n["arv_estimado"] == 584000.0, n
assert n["arv_fonte"] == "blend_mediana_predicao"
assert o["arv_estimado"] == 688000.0
print(f"  caso sintetico: max {o['arv_estimado']:,.0f} (ROI {o['roi_conservador']:+.2f}) "
      f"-> blend {n['arv_estimado']:,.0f} (ROI {n['roi_conservador']:+.2f})")
print("  para voltar: _restaurar_ancora()")


patch ativo: ARV = media(mediana do bairro, predicao do imovel)
  formula agora: (valor_m2_predicao + valor_m2_bairro) / 2 * metragem


AssertionError: 

In [ ]:
# RODA O LOTE COM O BLEND ATIVO, em projeto LangSmith proprio (baseline fica intacto).
import json
import time
from datetime import datetime, timedelta, timezone

LS_PROJETO_BASE = os.environ.get("LANGCHAIN_PROJECT", "Agente-Imovel")
LS_PROJETO_BLEND = os.environ.get("FLIP_LS_PROJETO_BLEND", "Agente-Imovel-Atiradores-Blend")
N_BLEND = int(os.environ.get("FLIP_LOTE_PILOTO") or len(PAYLOADS_ATIRADORES))
assert tf.calcular_metricas_flip is _calcular_metricas_blend, "rode a celula do patch antes"

PROJ_ANTERIOR = os.environ.get("LANGCHAIN_PROJECT")
TAG_ANTERIOR = os.environ.get("FLIP_LOTE_TAG")
os.environ["LANGCHAIN_PROJECT"] = LS_PROJETO_BLEND
os.environ["FLIP_LOTE_TAG"] = "blend"
print(f"experimento: projeto {LS_PROJETO_BLEND} (tag lote:blend) | baseline em {LS_PROJETO_BASE}")

resultados_blend = []
try:
    for i, payload in enumerate(PAYLOADS_ATIRADORES[:N_BLEND], 1):
        linha = SELECAO_ATIRADORES.iloc[payload["_idx"]]
        t0 = time.perf_counter()
        saida = await avaliar_imovel_agente(payload, max_tentativas=2)
        metricas = saida.graders.get("metricas_tool") or {}
        dur = time.perf_counter() - t0
        resultados_blend.append({
            "imovel": payload["_id"], "ppm2": round(float(linha["ppm2"])),
            "arv_blend": metricas.get("arv_estimado"),
            "arv_fonte": metricas.get("arv_fonte"),
            "roi": metricas.get("roi_conservador"),
            "flip": str(saida.potencial_house_flip), "score": saida.score_potencial_flip,
            "aprovado": saida.aprovado_graders,
            "justificativa": saida.justificativa_potencial,
            "dur_s": round(dur, 1),
        })
        print(f"{i:2d}/{N_BLEND} {payload['_id']} ppm2 {linha['ppm2']:7.0f} -> "
              f"{saida.potencial_house_flip} | score {saida.score_potencial_flip} "
              f"| roi {metricas.get('roi_conservador')} | aprov {saida.aprovado_graders} "
              f"| {dur:.0f}s", flush=True)
finally:
    try:
        from langchain_core.tracers.langchain import wait_for_all_tracers

        wait_for_all_tracers()
    except Exception:  # noqa: BLE001
        pass

    def _raiz_por_imovel(projeto, janela_h=48):
        """Uma varredura: imovel -> trace MAIS NOVA (a raiz do run atual)."""
        desde = datetime.now(timezone.utc) - timedelta(hours=janela_h)
        raizes = [r for r in ls.list_runs(project_name=projeto, start_time=desde)
                  if r.parent_run_id is None]
        por_imovel = {}
        for r in sorted(raizes, key=lambda x: x.start_time or datetime.min.replace(tzinfo=timezone.utc)):
            meta = (r.extra or {}).get("metadata") or {}
            nome = str(meta.get("imovel") or "")
            if not nome:
                import re as _re

                mm = _re.match(r"auto-(.+)-[0-9a-f]{10}$", str(meta.get("thread_id") or ""))
                nome = mm.group(1) if mm else ""
            if nome:
                por_imovel[nome] = r
        return por_imovel

    def _gravar_feedback_blend(por_imovel):
        gravados = 0
        for r in resultados_blend:
            traco = por_imovel.get(r["imovel"])
            if traco is None:
                print(f"  (sem trace para {r['imovel']}: feedback nao gravado)")
                continue
            for chave, valor, comentario in (
                ("roi_conservador", r["roi"] or 0, None),
                ("score_potencial_flip", r["score"] or 0, None),
                ("aprovado_graders", 1 if r["aprovado"] else 0, None),
                ("potencial_house_flip", 1 if r["flip"] == "True" else 0, r["flip"]),
                ("veredito", 1, json.dumps(r, ensure_ascii=False, default=str)),
            ):
                try:
                    ls.create_feedback(str(traco.id), key=chave,
                                       score=float(valor), comment=comentario)
                except Exception as exc:  # noqa: BLE001
                    print(f"  feedback {chave} falhou:", type(exc).__name__)
            gravados += 1
        return gravados

    if resultados_blend:
        gravados = _gravar_feedback_blend(_raiz_por_imovel(LS_PROJETO_BLEND))
        print(f"feedback do blend gravado em {gravados} de {len(resultados_blend)} traces")

os.environ["LANGCHAIN_PROJECT"] = PROJ_ANTERIOR or LS_PROJETO_BASE
if TAG_ANTERIOR is None:
    os.environ.pop("FLIP_LOTE_TAG", None)
else:
    os.environ["FLIP_LOTE_TAG"] = TAG_ANTERIOR

tabela_blend = pd.DataFrame(resultados_blend)
print()
print(tabela_blend.to_string(index=False))


In [ ]:
tabela_blend

,imovel,ppm2,arv_blend,arv_fonte,roi,flip,score,aprovado,justificativa,dur_s
0,atiradores-01,2962,1249402.00,None,0.3972,True,9.0,True,O imóvel atende à regra de 15% de lucro projet...,180.4
1,atiradores-02,3179,1785222.98,None,0.4284,True,9.0,True,O imóvel atende à regra de 15% de lucro projet...,163.2
2,atiradores-03,3499,1358196.40,None,0.5562,True,9.0,True,O imóvel atende à regra de 15% de lucro projet...,27.6
3,atiradores-04,3846,852656.81,None,0.3153,True,9.0,True,O ROI conservador de 31.53% supera significati...,26.0
4,atiradores-05,4012,1237717.26,None,0.3399,True,9.0,True,O imóvel atende ao critério de potencial de fl...,35.5
5,atiradores-06,4211,773215.92,None,0.3835,True,8.5,True,O imóvel atende ao critério de rentabilidade d...,44.9
6,atiradores-07,4455,766673.33,None,0.1967,True,9.0,True,O imóvel atende ao critério mínimo de ROI (19....,85.4
7,atiradores-08,4537,993339.89,None,0.4931,True,9.0,True,O imóvel atende ao critério de potencial de fl...,287.9
8,atiradores-09,4583,948984.00,None,0.2256,True,9.0,True,O ROI conservador de 22.56% é superior ao míni...,31.4
9,atiradores-10,4652,2338770.59,None,0.0871,False,3.0,True,O imóvel não atende à regra de 15% de lucro pr...,44.0


In [ ]:
# COMPARACAO: baseline (conta original) x blend, lidos do feedback de cada projeto.
# A justificativa do BASELINE vem da saida da tool registrar_decisao (na arvore);
# a do BLEND ficou em memoria na celula anterior (resultados_blend).
import json
import re
from datetime import datetime, timedelta, timezone

LS_PROJETO_BASE = os.environ.get("LANGCHAIN_PROJECT", "Agente-Imovel")
LS_PROJETO_BLEND = os.environ.get("FLIP_LS_PROJETO_BLEND", "Agente-Imovel-Atiradores-Blend")


def _feedback_por_imovel(projeto, janela_h=72):
    """imovel -> {chave do feedback: score ou comentario} na trace mais recente."""
    desde = datetime.now(timezone.utc) - timedelta(hours=janela_h)
    raizes = [r for r in ls.list_runs(project_name=projeto, start_time=desde)
              if r.parent_run_id is None]
    por_imovel = {}
    for r in sorted(raizes, key=lambda x: x.start_time or datetime.min.replace(tzinfo=timezone.utc)):
        meta = (r.extra or {}).get("metadata") or {}
        nome = str(meta.get("imovel") or "")
        if not nome:
            m = re.match(r"auto-(.+)-[0-9a-f]{10}$", str(meta.get("thread_id") or ""))
            nome = m.group(1) if m else ""
        if nome:
            por_imovel[nome] = r
    saida = {}
    for nome, run in por_imovel.items():
        fb = {}
        try:
            for f in ls.list_feedback(run_ids=[str(run.id)]):
                fb[f.key] = f.score if f.score is not None else f.comment
        except Exception as exc:  # noqa: BLE001
            fb["_erro"] = f"{type(exc).__name__}: {str(exc)[:80]}"
        saida[nome] = fb
    return saida


def _justificativa_baseline(projeto, run, janela_h=72):
    """Texto do veredito do agente na conta ORIGINAL: saida de registrar_decisao."""
    try:
        filhos = [t for t in ls.list_runs(project_name=projeto, parent_run_id=run.id)
                  if t.run_type == "tool" and t.name == "registrar_decisao"]
        if not filhos:
            return "(sem registrar_decisao na arvore)"
        saida = (max(filhos, key=lambda t: t.start_time or datetime.min).outputs or {})
        texto = json.dumps(saida, ensure_ascii=False, default=str)
        m = re.search(r'"justificativa_potencial"\s*:\s*"([^"]+)"', texto)
        return (m.group(1) if m else texto[:400])[:600]
    except Exception as exc:  # noqa: BLE001
        return f"(falhou: {type(exc).__name__})"


base_fb = _feedback_por_imovel(LS_PROJETO_BASE)
blend_fb = _feedback_por_imovel(LS_PROJETO_BLEND)
linhas = []
for imovel in sorted(set(base_fb) | set(blend_fb)):
    b, n = base_fb.get(imovel, {}), blend_fb.get(imovel, {})
    linhas.append({
        "imovel": imovel,
        "roi_base": b.get("roi_conservador"), "roi_blend": n.get("roi_conservador"),
        "flip_base": b.get("potencial_house_flip"), "flip_blend": n.get("potencial_house_flip"),
        "score_base": b.get("score_potencial_flip"), "score_blend": n.get("score_potencial_flip"),
        "aprov_base": b.get("aprovado_graders"), "aprov_blend": n.get("aprovado_graders"),
    })
cmp_df = pd.DataFrame(linhas)
if len(cmp_df):
    print("=== (1) vereditos lado a lado (feedback de cada projeto) ===")
    print(cmp_df.to_string(index=False))
    flip_b = int((cmp_df["flip_base"] == 1).sum())
    flip_n = int((cmp_df["flip_blend"] == 1).sum())
    print(f"\nflip True: baseline {flip_b} | blend {flip_n} | "
          f"mudou de flip para nao-flip: {int(((cmp_df['flip_base']==1)&(cmp_df['flip_blend']==0)).sum())}")

    print("\n=== (2) as respostas do agente (justificativa) ===")
    for _, row in cmp_df.iterrows():
        imovel = row["imovel"]
        run = None
        desde = datetime.now(timezone.utc) - timedelta(hours=72)
        raizes = [r for r in ls.list_runs(project_name=LS_PROJETO_BASE, start_time=desde)
                  if r.parent_run_id is None]
        for r in sorted(raizes, key=lambda x: x.start_time or datetime.min.replace(tzinfo=timezone.utc)):
            meta = (r.extra or {}).get("metadata") or {}
            if str(meta.get("imovel") or "") == imovel:
                run = r
        j_base = _justificativa_baseline(LS_PROJETO_BASE, run) if run is not None else "(sem trace)"
        j_blend = ""
        for r in resultados_blend:
            if r["imovel"] == imovel:
                j_blend = r["justificativa"]
                break
        print(f"\n--- {imovel} (flip base {row['flip_base']} -> blend {row['flip_blend']}) ---")
        print("  BASE :", (j_base or "(vazio)").replace("\n", " ")[:400])
        print("  BLEND:", (j_blend or "(vazio)").replace("\n", " ")[:400])
else:
    print("(sem feedback nas duas janelas: rode as celulas do lote e do experimento)")

_restaurar_ancora()


C:\Users\jefer\AppData\Local\Temp\ipykernel_2824\3884788469.py:15: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  raizes = [r for r in ls.list_runs(project_name=projeto, start_time=desde)


=== (1) vereditos lado a lado (feedback de cada projeto) ===
       imovel  roi_base  roi_blend  flip_base  flip_blend  score_base  score_blend  aprov_base  aprov_blend
atiradores-01    0.0000     0.3972        1.0         1.0         9.0          9.0         0.0          1.0
atiradores-02    1.3850     0.4283        1.0         1.0         9.0          7.0         1.0          1.0
atiradores-03    1.1663     0.5562        1.0         1.0         9.0          9.0         1.0          1.0
atiradores-04    0.9709     0.3153        1.0         1.0         9.0          9.0         1.0          1.0
atiradores-05    0.8893     0.3399        1.0         1.0         9.0          9.0         1.0          1.0
atiradores-06    0.8003     0.3835        1.0         1.0         8.5          9.0         1.0          1.0
atiradores-07    0.7014     0.1967        1.0         1.0         8.5          9.0         1.0          1.0
atiradores-08    0.6746     0.4931        1.0         1.0         9.0      

C:\Users\jefer\AppData\Local\Temp\ipykernel_2824\3884788469.py:79: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  raizes = [r for r in ls.list_runs(project_name=LS_PROJETO_BASE, start_time=desde)
C:\Users\jefer\AppData\Local\Temp\ipykernel_2824\3884788469.py:41: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  filhos = [t for t in ls.list_runs(project_name=projeto, parent_run_id=run.id)



--- atiradores-01 (flip base 1.0 -> blend 1.0) ---
  BASE : (sem registrar_decisao na arvore)
  BLEND: O imóvel atende à regra de 15% de lucro projetado (passa_regra_15pct=True), com ROI conservador de 39.72%, indicando bom potencial de flip.

--- atiradores-02 (flip base 1.0 -> blend 1.0) ---
  BASE : (sem registrar_decisao na arvore)
  BLEND: O imóvel atende à regra de 15% de lucro projetado (passa_regra_15pct=True), apresentando um ROI conservador de 42.84%. A cobertura possui características de alto valor agregado (duplex, terraço com spa/bar/escritório) e está em um bairro desejado (Atiradores, Joinville).

--- atiradores-03 (flip base 1.0 -> blend 1.0) ---
  BASE : (sem registrar_decisao na arvore)
  BLEND: O imóvel atende à regra de 15% de lucro projetado (ROI conservador de 55.62%) e possui métricas favoráveis, apesar do desgaste leve aparente nas fotos.

--- atiradores-04 (flip base 1.0 -> blend 1.0) ---
  BASE : (sem registrar_decisao na arvore)
  BLEND: O ROI conservador de

In [ ]:
cmp_df

,imovel,roi_base,roi_blend,flip_base,flip_blend,score_base,score_blend,aprov_base,aprov_blend
0,atiradores-01,0.0000,0.3972,1.0,1.0,9.0,9.0,0.0,1.0
1,atiradores-02,1.3850,0.4283,1.0,1.0,9.0,7.0,1.0,1.0
2,atiradores-03,1.1663,0.5562,1.0,1.0,9.0,9.0,1.0,1.0
3,atiradores-04,0.9709,0.3153,1.0,1.0,9.0,9.0,1.0,1.0
4,atiradores-05,0.8893,0.3399,1.0,1.0,9.0,9.0,1.0,1.0
5,atiradores-06,0.8003,0.3835,1.0,1.0,8.5,9.0,1.0,1.0
6,atiradores-07,0.7014,0.1967,1.0,1.0,8.5,9.0,1.0,1.0
7,atiradores-08,0.6746,0.4931,1.0,1.0,9.0,9.0,1.0,1.0
8,atiradores-09,0.6539,0.2256,1.0,1.0,9.0,9.0,1.0,1.0
9,atiradores-10,0.0000,0.0871,0.0,0.0,0.0,3.0,0.0,1.0


## Resultados do workflow (CI)

O workflow `.github/workflows/teste-flip-react.yml` grava o lote em
`dados/joinville/joinville_avaliacao_flip_<MES>_0_<limite>_<bairro>.parquet`
e publica o mesmo arquivo como artefato `teste-flip-react-<bairro>`.

Esta secao nao roda o agente: ela le o resultado do CI e audita o que veio.

- **Offline:** baixe o artefato na pagina do run, extraia e aponte
  `FLIP_RESULTADO_CI` para o `.parquet` (ou deixe o arquivo em `dados/joinville/`).
- **API:** defina `RUN_ID` na celula abaixo e ligue `BAIXAR_ARTEFATO_CI = True`.


In [ ]:
# Download opcional do artefato do CI. Desligado por default: offline basta
# apontar FLIP_RESULTADO_CI para o parquet extraido do artefato.
import os
import zipfile
from pathlib import Path

BAIXAR_ARTEFATO_CI = False
RUN_ID = os.environ.get("FLIP_CI_RUN_ID", "").strip()  # ex.: "44811991544"
REPO_CI = "Jeferson100/Zap-Imoveis-Web-Scraping"


def _raiz_repo() -> Path:
    """Raiz do repo a partir do diretorio atual (funciona de Notebooks/)."""
    aqui = Path.cwd().resolve()
    for pasta in (aqui, *aqui.parents):
        if (pasta / "pyproject.toml").exists():
            return pasta
    return aqui


if BAIXAR_ARTEFATO_CI:
    import requests

    assert RUN_ID, "defina RUN_ID (ou FLIP_CI_RUN_ID) com o id do run do GitHub Actions"
    token = ""
    env = _raiz_repo() / ".env"
    if env.exists():
        for linha in env.read_text(encoding="utf-8").splitlines():
            if linha.startswith("ACESSO_TOKEN_GITHUB"):
                token = linha.split("=", 1)[1].strip().strip('"').strip("'")
                break
    assert token, "ACESSO_TOKEN_GITHUB ausente no .env da raiz"

    cab = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"}
    api = f"https://api.github.com/repos/{REPO_CI}/actions/runs/{RUN_ID}/artifacts"
    artefatos = requests.get(api, headers=cab, timeout=60).json().get("artifacts") or []
    alvo = next((a for a in artefatos if a["name"].startswith("teste-flip-react")), None)
    assert alvo, f"artefato teste-flip-react-* nao encontrado no run {RUN_ID}"
    conteudo = requests.get(alvo["archive_download_url"], headers=cab, timeout=120).content
    # O zip guarda caminhos relativos a raiz (dados/joinville/..., logs/...).
    destino = _raiz_repo() / "dados" / "joinville" / "_artefato_ci.zip"
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(conteudo)
    with zipfile.ZipFile(destino) as z:
        z.extractall(_raiz_repo())
    destino.unlink()
    print("artefato do run", RUN_ID, "extraido em", _raiz_repo())
else:
    print("download desligado (offline): use FLIP_RESULTADO_CI ou deixe o parquet em dados/joinville/")


download desligado (offline): use FLIP_RESULTADO_CI ou deixe o parquet em dados/joinville/


In [21]:
# Carrega o parquet do run do CI.
import glob
import os
from pathlib import Path

import pandas as pd


def _raiz_repo() -> Path:
    aqui = Path.cwd().resolve()
    for pasta in (aqui, *aqui.parents):
        if (pasta / "pyproject.toml").exists():
            return pasta
    return aqui


RAIZ = _raiz_repo()
caminho = os.environ.get("FLIP_RESULTADO_CI", "").strip()
if not caminho:
    candidatos = sorted(
        glob.glob(str(RAIZ / "dados/joinville/avaliacao_flip_2026-11_atiradores.parquet")),
        key=os.path.getmtime,
    )
    caminho = candidatos[-1] if candidatos else ""
assert caminho and Path(caminho).exists(), (
    "parquet do CI nao encontrado. Baixe o artefato do run (Actions > run > "
    "Artifacts > teste-flip-react-<bairro>), extraia e defina "
    "FLIP_RESULTADO_CI=/caminho/do/arquivo.parquet"
)

df_ci = pd.read_parquet(caminho)
print("arquivo:", Path(caminho).name)
print("linhas:", len(df_ci), "| colunas:", list(df_ci.columns))


arquivo: avaliacao_flip_2026-11_atiradores.parquet
linhas: 407 | colunas: ['url', 'score_potencial_flip', 'potencial_house_flip', 'justificativa', 'riscos', 'recomendacoes', 'observacoes', 'erro', 'valor_m2_blend', 'arv_fonte', 'aprovado_graders', 'tentativas', 'ambiente_fotos', 'conservacao_fotos', 'metricas_base']


In [22]:
df_ci

,url,score_potencial_flip,potencial_house_flip,justificativa,riscos,recomendacoes,observacoes,erro,valor_m2_blend,arv_fonte,aprovado_graders,tentativas,ambiente_fotos,conservacao_fotos,metricas_base
0,https://www.chavesnamao.com.br/imovel/apartame...,3.0,False,O imóvel apresenta excelente conservação (scor...,"[""Preço de compra e preço por m² não informado...","[""Solicitar preço real de compra e comparar co...",A alta pontuação de conservação e potencial de...,None,NaN,None,None,NaN,None,None,None
1,https://www.chavesnamao.com.br/imovel/apartame...,5.0,False,"Apesar da excelente conservação (9.5), acabame...","[""Valor de compra inexistente (0.0)"", ""Preço p...","[""Solicitar o preço real de aquisição antes de...",A excelência física do imóvel indica alto pote...,None,NaN,None,None,NaN,None,None,None
2,https://www.chavesnamao.com.br/imovel/apartame...,6.5,False,"O preço por m² pago de R$6.997,73 está aproxim...","[""Possibilidade de defeitos ocultos (instalaçõ...","[""Buscar renegociação do preço de compra para ...",Embora o imóvel apresente excelente conservaçã...,None,NaN,None,None,NaN,None,None,None
3,https://www.chavesnamao.com.br/imovel/apartame...,3.5,False,O preço atual de R$ 778.000 equivale a R$ 9.67...,"[""Margem de lucro muito estreita"", ""ROI abaixo...","[""Reavaliar preço de compra ou buscar imóvel c...","Apesar da excelente conservação e acabamento, ...",None,NaN,None,None,NaN,None,None,None
4,https://www.chavesnamao.com.br/imovel/apartame...,9.2,True,"O imóvel está sendo vendido a R$ 36,36 por m²,...","[""Volatilidade do mercado imobiliário local qu...","[""Realizar auditoria detalhada de custos de co...",O preço de compra extremamente abaixo do valor...,None,NaN,None,None,NaN,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
402,https://sc.olx.com.br/norte-de-santa-catarina/...,3.0,False,Regra dos 15% nao atendida - preco de entrada ...,"[""Preco acima do esperado para area"", ""Condomi...","[""Evitar compra para flip neste preco"", ""Busca...","Metragem 54m2 validada, valor R$ 340k contra A...",None,8887.97,blend_mediana_predicao,False,3.0,desconhecido,indefinido,bruto_sem_validacao
403,https://sc.olx.com.br/norte-de-santa-catarina/...,0.0,False,Veredito sintetizado pelo runner (loop encerra...,"[""veredito sintetizado: sem justificativa vali...","[""revisar manualmente""]",motivo_fim=justificativa vazia do modelo,None,8638.66,blend_mediana_predicao,False,3.0,desconhecido,indefinido,validado_sem_correcao
404,https://www.chavesnamao.com.br/imovel/apartame...,7.5,True,Apartamento 216m² no Atiradores/Joinville (áre...,"[""Reforma em apartamento alto padrão pode exig...","[""Negociar entrada até R$ 1.378M para manter m...",Valor do imóvel confirmado em R$ 1.365.003 via...,None,9819.67,blend_mediana_predicao,False,3.0,desconhecido,indefinido,validado_sem_correcao
405,https://sc.olx.com.br/norte-de-santa-catarina/...,3.0,False,ARV R$ 481.170 (blend predição R$ 7.105/m² + m...,"[""Preço de compra 18,6% acima do teto que fech...","[""Não prosseguir no preço atual (R$ 359.999)"",...",Dados validados e consistentes (confiança 0.9)...,None,8592.31,blend_mediana_predicao,False,3.0,desconhecido,indefinido,validado_sem_correcao


In [ ]:
# Resumo do run: distribuicoes e o que cada coluna revela.
def _col(nome):
    if nome in df_ci:
        return df_ci[nome]
    return pd.Series([None] * len(df_ci), index=df_ci.index)


def _vc(nome):
    return _col(nome).value_counts(dropna=False).to_dict()


resumo = pd.Series({
    "imoveis": len(df_ci),
    "erros": int(_col("erro").notna().sum()),
    "vereditos": _vc("potencial_house_flip"),
    "aprovado_graders": _vc("aprovado_graders"),
    "arv_fonte": _vc("arv_fonte"),
    "metricas_base": _vc("metricas_base"),
    "tentativas": _vc("tentativas"),
    "ambiente_fotos": _vc("ambiente_fotos"),
    "conservacao_fotos": _vc("conservacao_fotos"),
    "sintetizados": int(_col("observacoes").astype(str).str.contains("motivo_fim").sum()),
    "timeouts": int(_col("observacoes").astype(str).str.contains("timeout", case=False).sum()),
}, name="valor")
resumo.to_frame()


,valor
imoveis,10
erros,0
vereditos,"{'True': 7, 'False': 3}"
aprovado_graders,{False: 10}
arv_fonte,{'blend_mediana_predicao': 10}
metricas_base,"{'validado': 5, 'validado_sem_correcao': 3, 'b..."
tentativas,{3: 10}
ambiente_fotos,{'desconhecido': 10}
conservacao_fotos,{'indefinido': 10}
sintetizados,0


In [ ]:
# Uma linha por imovel, com as flags que explicam o veredito.
def _flags(r):
    marcas = []
    obs = str(r.get("observacoes") or "").lower()
    if "timeout" in obs:
        marcas.append("timeout")
    if "motivo_fim" in obs:
        marcas.append("sintetizado")
    if str(r.get("metricas_base") or "") == "bruto_sem_validacao":
        marcas.append("bruto_sem_validacao")
    if r.get("aprovado_graders") is False:
        marcas.append("graders_reprovado")
    return ", ".join(marcas)


tabela_ci = pd.DataFrame({
    "imovel": _col("url").astype(str)
        .str.replace(r"^https?://(www\.)?", "", regex=True).str[:52],
    "veredito": _col("potencial_house_flip"),
    "score": _col("score_potencial_flip"),
    "blend_m2": _col("valor_m2_blend").round(0),
    "arv_fonte": _col("arv_fonte"),
    "base": _col("metricas_base"),
    "tent": _col("tentativas"),
    "flags": df_ci.apply(_flags, axis=1),
})
tabela_ci


,imovel,veredito,score,blend_m2,arv_fonte,base,tent,flags
0,chavesnamao.com.br/imovel/apartamento-a-venda-...,False,2.0,10030.0,blend_mediana_predicao,validado,3,graders_reprovado
1,sc.olx.com.br/norte-de-santa-catarina/imoveis/...,True,7.0,6949.0,blend_mediana_predicao,bruto_sem_validacao,3,"bruto_sem_validacao, graders_reprovado"
2,zapimoveis.com.br/imovel/venda-apartamento-3-q...,True,8.0,7563.0,blend_mediana_predicao,validado,3,graders_reprovado
3,zapimoveis.com.br/imovel/venda-cobertura-2-qua...,True,8.0,6918.0,blend_mediana_predicao,bruto_sem_validacao,3,"bruto_sem_validacao, graders_reprovado"
4,chavesnamao.com.br/imovel/cobertura-a-venda-2-...,True,8.0,6796.0,blend_mediana_predicao,validado,3,graders_reprovado
5,zapimoveis.com.br/imovel/venda-apartamento-2-q...,False,2.0,10404.0,blend_mediana_predicao,validado,3,graders_reprovado
6,zapimoveis.com.br/imovel/venda-apartamento-3-q...,True,8.0,7441.0,blend_mediana_predicao,validado_sem_correcao,3,graders_reprovado
7,zapimoveis.com.br/imovel/venda-cobertura-2-qua...,True,8.0,8000.0,blend_mediana_predicao,validado_sem_correcao,3,graders_reprovado
8,sc.olx.com.br/norte-de-santa-catarina/imoveis/...,True,8.0,8177.0,blend_mediana_predicao,validado_sem_correcao,3,graders_reprovado
9,sc.olx.com.br/norte-de-santa-catarina/imoveis/...,False,5.0,7407.0,blend_mediana_predicao,validado,3,graders_reprovado


In [ ]:
# Checagens de qualidade: o que derrubava o teste do CI antes de virar codigo.
problemas = []


def _flag(msg):
    problemas.append(msg)


if _col("erro").notna().any():
    _flag(f"imoveis com erro: {_col('erro').dropna().astype(str).tolist()[:3]}")

vereditos = set(_col("potencial_house_flip").dropna().astype(str))
if not vereditos or not vereditos <= {"True", "False"}:
    _flag(f"vereditos invalidos: {vereditos}")

permitidas = {"blend_mediana_predicao", "predicao_sem_bairro", "bairro_sem_predicao"}
ruins = sorted(set(_col("arv_fonte").dropna().astype(str)) - permitidas)
if ruins:
    _flag(f"arv_fonte inesperada: {ruins}")

just = _col("justificativa").astype(str).str.strip()
curtas = just[just.str.strip(".").str.len() < 10]
if len(curtas):
    _flag(f"justificativas vazias/curtas: {len(curtas)} (linhas {list(curtas.index)})")

if not _col("aprovado_graders").dropna().astype(bool).any():
    _flag("nenhum imovel com aprovado_graders=True: G1 segue sem os pesos do ConvNeXt")

sem_validacao = int(_col("metricas_base").astype(str).eq("bruto_sem_validacao").sum())
if sem_validacao:
    _flag(f"{sem_validacao} imovel(is) calcularam metricas antes da validacao (G6 reprova)")

print("problemas:", len(problemas))
for p in problemas:
    print(" -", p)


problemas: 2
 - nenhum imovel com aprovado_graders=True: G1 segue sem os pesos do ConvNeXt
 - 2 imovel(is) calcularam metricas antes da validacao (G6 reprova)


In [ ]:
# Comparacao opcional com outro run: FLIP_RESULTADO_CI_ANTERIOR=/caminho/parquet
import os
from pathlib import Path

anterior = os.environ.get("FLIP_RESULTADO_CI_ANTERIOR", "").strip()
if anterior and Path(anterior).exists():
    df_anterior = pd.read_parquet(anterior)

    def _resumo(df):
        obs = df.get("observacoes", pd.Series(dtype=str)).astype(str)
        return {
            "imoveis": len(df),
            "vereditos": df.get("potencial_house_flip", pd.Series(dtype=str))
                .value_counts(dropna=False).to_dict(),
            "sintetizados": int(obs.str.contains("motivo_fim").sum()),
            "timeouts": int(obs.str.contains("timeout", case=False).sum()),
            "aprovado_graders": df.get("aprovado_graders", pd.Series(dtype=bool))
                .value_counts(dropna=False).to_dict(),
        }

    pd.DataFrame({"anterior": _resumo(df_anterior), "atual": _resumo(df_ci)})
else:
    print("sem comparacao: defina FLIP_RESULTADO_CI_ANTERIOR para comparar com outro parquet")


sem comparacao: defina FLIP_RESULTADO_CI_ANTERIOR para comparar com outro parquet
